In [ ]:
# รันเซลล์นี้บน Colab เท่านั้น: ในเครื่องใช้ venv เดิมและไม่ติดตั้งเพิ่ม
import sys, subprocess
REQUIREMENTS = 'numpy==2.4.6\npandas==3.0.6\nscikit-learn==1.9.1\nscipy==1.17.1\nthreadpoolctl==3.7.0\nlightgbm==4.7.0\ntorch==2.14.1\nmlflow==3.17.1\nfolktables==0.0.12\nmatplotlib==3.11.2\npsutil==7.2.2\npytest==9.1.1\nunsupervised-bias-detection==0.2.6\ngradio==6.30.0\njoblib==1.6.0\nkmodes==0.12.2\n'
all_pins = REQUIREMENTS.split()
hbac_pins = [p for p in all_pins if p.startswith('unsupervised-bias-detection==')]
pins = [p for p in all_pins if p not in hbac_pins]
# Colab โหลด numpy/pandas/torch ฯลฯ ไว้ในหน่วยความจำตั้งแต่เปิด runtime ถ้าติดตั้งเวอร์ชันอื่นทับ
# จะเกิด ImportError ปะปนสองเวอร์ชัน (เช่น numpy._core.umath) จนกว่าจะรีสตาร์ต จึงคงเวอร์ชันของ Colab ไว้
# และล็อกไม่ให้ pip เปลี่ยน ส่วนแพ็กเกจอื่นติดตั้งตาม requirements · เวอร์ชันจริงบันทึกใน results/environment.json
import importlib.metadata as md, tempfile
KEEP = {'numpy', 'pandas', 'scipy', 'scikit-learn', 'torch', 'matplotlib', 'joblib', 'threadpoolctl', 'psutil'}
def installed(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None
keep = {p.split('==')[0]: installed(p.split('==')[0]) for p in pins if p.split('==')[0] in KEEP}
constraints = tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False)
constraints.write('\n'.join(f'{n}=={v}' for n, v in keep.items() if v)); constraints.close()
to_install = [p for p in pins if not keep.get(p.split('==')[0])]
print('ใช้เวอร์ชันของ Colab:', {n: v for n, v in keep.items() if v})
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *to_install, '-c', constraints.name])
except subprocess.CalledProcessError:
    # ถ้าเวอร์ชันที่ล็อกไว้เข้ากับแพ็กเกจหลักของ Colab ไม่ได้ ให้ pip เลือกเวอร์ชันที่เข้ากันเอง (ยังล็อกแพ็กเกจหลัก)
    print('ติดตั้งตามเวอร์ชันที่ล็อกไม่ได้ ใช้เวอร์ชันที่เข้ากับ Colab แทน')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *[p.split('==')[0] for p in to_install], '-c', constraints.name])
# HBAC 0.2.6 ระบุ numpy<2 ใน metadata แต่ทดสอบกับ numpy 2 แล้ว จึงติดตั้งแบบ --no-deps
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', *hbac_pins])


# ต้นแบบนำร่อง Fair Hidden Groups
เลือก Runtime → Run all บน CPU; ไม่ต้องอัปโหลด src หรือข้อมูลก่อน
HMDA GA 2023; ประชากรใช้ประเมินและ oracle เท่านั้น · MLflow SQLite ไม่มี artifact รายรัน
รอบ 6 คง DRO และเพิ่ม PP-parity/EO บน ERM-LGBM/MLP: หาเกณฑ์และเลือกด้วย validation เท่านั้น พร้อม guard accuracy ลดไม่เกิน 2 จุดร้อยละ; ดูหัวข้อปรับเกณฑ์รายกลุ่มในสรุปและผล PP ท้ายหน้าสาธิต


In [ ]:
# ปล่อยว่างได้: notebook จะสร้างโค้ด src ให้ครบเอง
REPO_URL = "https://github.com/ronnachaii/fair-credit-hidden-groups.git"
QUICK = True
from pathlib import Path
import subprocess, os
BASE = Path('/content/rsu_fair_pilot') if Path('/content').exists() else Path.cwd() / 'colab_pilot'
if REPO_URL and not BASE.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(BASE)])
BASE.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
for folder in ['src/fairgroups', 'tests', 'results']:
    Path(folder).mkdir(parents=True, exist_ok=True)
# ไฟล์ embedded ด้านล่างเป็นเวอร์ชันเดียวกับชุดส่งงาน แม้เลือก clone repo


In [ ]:
%%writefile requirements.txt
numpy==2.4.6
pandas==3.0.6
scikit-learn==1.9.1
scipy==1.17.1
threadpoolctl==3.7.0
lightgbm==4.7.0
torch==2.14.1
mlflow==3.17.1
folktables==0.0.12
matplotlib==3.11.2
psutil==7.2.2
pytest==9.1.1
unsupervised-bias-detection==0.2.6
gradio==6.30.0
joblib==1.6.0
kmodes==0.12.2


In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
addopts = --basetemp=.pytest_tmp


In [ ]:
%%writefile config.py
"""ค่าทดลองชุดเดียวกันสำหรับเครื่องส่วนตัวและ Colab CPU"""
from dataclasses import asdict, dataclass
import sys


@dataclass
class Config:
    quick: bool = True
    dataset: str = "hmda"
    state: str = "GA"
    lei_top_n: int = 30
    jtt_grid: tuple = (1.0, 2.0, 4.0)
    seeds: tuple = (17, 29, 43)
    label_fractions: tuple = (1.0,)
    ks: tuple = (2, 4, 8)
    max_rows: int = 20000
    epochs: int = 10
    scarf_epochs: int = 8
    batch_size: int = 256
    hidden: int = 64
    embedding: int = 32
    learning_rate: float = 0.002
    weight_decay: float = 0.001
    dro_eta: float = 0.05
    dro_adjustment: float = 0.0
    dro_eta_grid: tuple = (0.01, 0.1)
    dro_adjustment_grid: tuple = (0.0, 1.0)
    min_dro_group: int = 50
    dro_finetune_epochs: int = 5
    # งบ 5 epoch ครอบคลุม checkpoint 1–5 อยู่แล้ว ไม่ฝึกงบ 3 ซ้ำ
    dro_epoch_grid: tuple = (5,)
    dro_variants: tuple = ('cy', 'cy_bal', 'c', 'c_bal')
    selection_accuracy_tolerance: float = 0.02
    dro_learning_rate_grid: tuple = (0.0002, 0.001)
    jtt_upweight: float = 10.0
    jtt_identification_epochs: int = 3
    corruption_rate: float = 0.6
    temperature: float = 0.5
    oof_folds: int = 5
    trees: int = 100
    min_category_frequency: int = 20
    min_fair_group: int = 30
    # macOS env นี้มี libomp หลายชุด ใช้หนึ่งเธรดเพื่อเลี่ยงการชนกัน
    threads: int = 1 if sys.platform == "darwin" else 2

    def to_dict(self):
        return asdict(self)


def make_config(quick=True):
    """โหมดเต็มใช้ข้อมูลทั้งหมดและเพิ่มจำนวนรอบฝึก"""
    cfg = Config(quick=quick)
    if not quick:
        cfg.seeds = (17, 29, 43, 59, 71)
        cfg.label_fractions = (1.0,)
        cfg.max_rows = 0
        cfg.epochs = 40
        cfg.scarf_epochs = 30
        cfg.hidden = 128
        cfg.trees = 300
    return cfg


In [ ]:
%%writefile run_pilot.py
"""จุดเริ่มต้นสำหรับ command line และ notebook"""
import argparse
import sys
import os
from pathlib import Path

if sys.platform == "darwin":
    os.environ["OMP_NUM_THREADS"] = "1"
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))
from config import make_config
from fairgroups.run import run_experiment
from threadpoolctl import threadpool_limits


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="HMDA hidden-group fairness pilot (CPU)")
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--quick", action="store_true", help="3 seeds, full labels, 20,000 rows")
    mode.add_argument("--full", action="store_true", help="5 seeds, full labels, all rows")
    parser.add_argument("--dataset", choices=["hmda", "acs"], default="hmda")
    parser.add_argument("--state", default="GA")
    args = parser.parse_args()
    config = make_config(quick=not args.full)
    config.dataset, config.state = args.dataset, args.state.upper()
    if args.dataset == "acs":
        from fairgroups.acs_run import run_experiment
    with threadpool_limits(limits=config.threads):
        run_experiment(config, ROOT)


In [ ]:
%%writefile src/fairgroups/__init__.py
"""ต้นแบบค้นหากลุ่มและเรียนรู้ให้ทนต่อกลุ่มเสียเปรียบ"""


In [ ]:
%%writefile src/fairgroups/acs_run.py
"""รัน pilot ทุกวิธีและเก็บหลักฐานแยกข้อมูลฝึก/ทดสอบ"""
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from .data import (load_data, split_data, labeled_indices, TrainPreprocessor,
                   reference_groups, categorical_frames)
from .models import set_seed, train_mlp, predict_mlp, validation_scores
from .representations import train_scarf, encode_scarf, crossfit_error
from .clustering import discover, cross_label_groups
from .efficiency import ResourceMonitor, network_cost, inference_seconds, environment
from .metrics import evaluate
from .tracking import setup_tracking, log_row
from .reporting import generate_reports


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


def record_decision(rows, model, predict, selected, base, test, config, env, output):
    """test ใช้วัดผลเมื่อเลือก checkpoint ด้วย validation เสร็จแล้ว"""
    selection, epoch, score = selected
    row = dict(base, selection=selection, selected_epoch=epoch, validation_selection_score=score)
    row.update(evaluate(test["y"], predict(test["x"]), test["s"], config.min_fair_group))
    row["inference_seconds_per_1000"] = inference_seconds(predict, test["x"])
    if base["method"] == "ERM-LGBM":
        trees = model.booster_.dump_model(num_iteration=epoch)["tree_info"]
        row.update(parameters=np.nan, flops_per_row=np.nan,
                   tree_count=len(trees), leaf_count=sum(t["num_leaves"] for t in trees))
    else:
        row.update(network_cost(model), tree_count=np.nan, leaf_count=np.nan)
    row["mlflow_run_id"] = log_row(row, env["libraries"], config)
    rows.append(row)
    pd.DataFrame(rows).to_csv(output / "pilot_results.csv", index=False)


def run_experiment(config, root="."):
    """ต้นทางทั้งหมดเป็น relative path จึงใช้เหมือนกันใน Colab"""
    root = Path(root).resolve()
    output = root / "results" / "acs_optional"
    output.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    set_seed(config.seeds[0], config.threads)
    env = environment(config)
    save_json(output / "environment.json", env)
    setup_tracking(root)
    with ResourceMonitor() as overall:
        x, y, sensitive = load_data(root / "data")
        env["eligible_source_rows"] = len(y)
        rows, cluster_rows, resource_rows, audits = [], [], [], []
        for seed in config.seeds:
            print(f"seed={seed}: split / preprocess / SCARF", flush=True)
            set_seed(seed, config.threads)
            splits = split_data(x, y, sensitive, seed, config.max_rows)
            train, val, test = (splits[name] for name in ["train", "val", "test"])
            audit = {"seed": seed, "train_ids": train["ids"].tolist(),
                     "val_ids": val["ids"].tolist(), "test_ids": test["ids"].tolist(),
                     "features": list(train["x"].columns), "fits": [], "error_crossfits": []}
            def add_fit(stage, ids, **extra):
                assert set(ids).issubset(set(train["ids"]))
                assert set(ids).isdisjoint(test["ids"])
                audit["fits"].append({"stage": stage, "fit_ids": np.asarray(ids).tolist(), **extra})
            with ResourceMonitor() as pre_resource:
                processor = TrainPreprocessor(config.min_category_frequency).fit(train["x"])
                transformed = {name: processor.transform(part["x"]) for name, part in splits.items()}
                categorical = categorical_frames(splits)
            add_fit("onehot_scaler", processor.fit_ids_)
            add_fit("lgbm_categories", train["ids"])
            with ResourceMonitor() as scarf_resource:
                scarf = train_scarf(train["x"], processor, config, seed)
                scaler = StandardScaler()
                reps = {"raw": transformed["train"],
                        "scarf": scaler.fit_transform(encode_scarf(scarf, transformed["train"])).astype(np.float32)}
                # val/test ไม่เคยเข้า fit แม้ในการเรียนแบบไม่มีป้ายกำกับ
                scarf_holdout = {name: scaler.transform(encode_scarf(scarf, transformed[name]))
                                 for name in ["val", "test"]}
            add_fit("scarf", scarf.fit_ids_)
            add_fit("scarf_scaler", train["ids"])
            encoder_cost = network_cost(scarf.encoder)
            pretraining_cost = network_cost(scarf)
            resource_rows.append(dict(representation="scarf", discovery_label_fraction=0, seed=seed,
                discovery_train_seconds=scarf_resource.seconds, peak_memory_mb=scarf_resource.peak_mb,
                inference_seconds_per_1000=inference_seconds(lambda xx: encode_scarf(scarf, xx), transformed["test"]),
                **encoder_cost, pretraining_parameters=pretraining_cost["parameters"],
                pretraining_flops_per_row=pretraining_cost["flops_per_row"]))
            resource_rows.append(dict(representation="raw", discovery_label_fraction=0, seed=seed,
                discovery_train_seconds=pre_resource.seconds, peak_memory_mb=pre_resource.peak_mb,
                inference_seconds_per_1000=inference_seconds(processor.transform, test["x"]),
                parameters=0, flops_per_row=0, pretraining_parameters=0, pretraining_flops_per_row=0))
            discoveries = {}
            for rep_name, rep in reps.items():
                for k in config.ks:
                    with ResourceMonitor() as cluster_resource:
                        clusters, scores, km = discover(rep, train["ids"], train["s"], k, seed)
                        # ใช้เพียง transform/predict บน holdout เพื่อยืนยันขั้นนำไปใช้
                        for name in ["val", "test"]:
                            km.predict(transformed[name] if rep_name == "raw" else scarf_holdout[name])
                    add_fit(f"kmeans_{rep_name}_{k}", km.fit_ids_)
                    discovery_seconds = cluster_resource.seconds + (scarf_resource.seconds if rep_name == "scarf" else 0)
                    discovery_peak = max(cluster_resource.peak_mb, scarf_resource.peak_mb if rep_name == "scarf" else pre_resource.peak_mb)
                    discoveries[(rep_name, k)] = (clusters, scores, discovery_seconds, discovery_peak)
                    cluster_row = dict(representation=rep_name, k=k, seed=seed,
                        discovery_label_fraction=0, n_discovery=len(train["y"]), **scores,
                        discovery_train_seconds=discovery_seconds, peak_memory_mb=discovery_peak)
                    cluster_row["mlflow_run_id"] = log_row(cluster_row, env["libraries"], config, "clustering")
                    cluster_rows.append(cluster_row)
            vg = reference_groups(val["s"])["intersection"]
            for fraction in config.label_fractions:
                labeled = labeled_indices(train["y"], fraction, seed)
                xl, yl = transformed["train"][labeled], train["y"][labeled]
                ids = train["ids"][labeled]
                print(f"seed={seed}, labels={fraction:g}, n={len(yl)}: OOF + 16 methods", flush=True)
                with ResourceMonitor() as error_resource:
                    error_rep, error_audit = crossfit_error(xl, yl, ids, config, seed)
                error_audit["label_fraction"] = fraction
                audit["error_crossfits"].append(error_audit)
                add_fit(f"error_oof_and_scaler_{fraction}", ids)
                resource_rows.append(dict(representation="error", discovery_label_fraction=fraction, seed=seed,
                    discovery_train_seconds=error_resource.seconds, peak_memory_mb=error_resource.peak_mb,
                    inference_seconds_per_1000=np.nan,
                    parameters=np.nan, flops_per_row=np.nan, pretraining_parameters=np.nan,
                    pretraining_flops_per_row=np.nan))
                current_discoveries = dict(discoveries)
                for k in config.ks:
                    with ResourceMonitor() as er_cluster:
                        clusters, scores, km = discover(error_rep, ids, train["s"].iloc[labeled], k, seed)
                    add_fit(f"kmeans_error_{k}_{fraction}", km.fit_ids_)
                    seconds = error_resource.seconds + er_cluster.seconds
                    peak = max(error_resource.peak_mb, er_cluster.peak_mb)
                    current_discoveries[("error", k)] = (clusters, scores, seconds, peak)
                    cluster_row = dict(representation="error", k=k, seed=seed,
                        discovery_label_fraction=fraction, n_discovery=len(yl), **scores,
                        discovery_train_seconds=seconds, peak_memory_mb=peak)
                    cluster_row["mlflow_run_id"] = log_row(cluster_row, env["libraries"], config, "clustering")
                    cluster_rows.append(cluster_row)
                common = dict(seed=seed, label_fraction=fraction, n_labeled=len(yl), n_train=len(train["y"]),
                              n_val=len(val["y"]), n_test=len(test["y"]), input_dimension=xl.shape[1],
                              preprocessing_seconds=pre_resource.seconds,
                              threshold=0.5, label_budget_includes_discovery=True)
                # LightGBM ใช้ categorical โดยตรง เลือกจำนวนต้นไม้จากกริดบน validation
                with ResourceMonitor() as lgb_resource:
                    lgb = LGBMClassifier(n_estimators=config.trees, num_leaves=15, max_depth=5,
                        learning_rate=0.05, verbosity=-1, n_jobs=config.threads, random_state=seed,
                        deterministic=True, force_col_wise=True)
                    lgb.fit(categorical["train"].iloc[labeled], yl)
                    candidates = sorted(set([10, 25, 50, config.trees]))
                    scores_by_tree = {t: validation_scores(val["y"],
                        lgb.predict_proba(categorical["val"], num_iteration=t)[:, 1], vg) for t in candidates}
                base = dict(common, method="ERM-LGBM", representation="none", k=0,
                            decision_train_seconds=lgb_resource.seconds, discovery_train_seconds=0,
                            train_seconds=pre_resource.seconds + lgb_resource.seconds,
                            peak_memory_mb=max(pre_resource.peak_mb, lgb_resource.peak_mb))
                for selection in ["average", "worst_group"]:
                    epoch = max(candidates, key=lambda t: scores_by_tree[t][selection])
                    predict = lambda xx, epoch=epoch: lgb.predict_proba(xx, num_iteration=epoch)[:, 1]
                    record_decision(rows, lgb, predict, (selection, epoch, scores_by_tree[epoch][selection]),
                        base, dict(test, x=categorical["test"]), config, env, output)
                # JTT รอบระบุตัวอย่างยากมีจำนวน epoch คงที่ ไม่ใช้กลุ่มจริงเลือก
                with ResourceMonitor() as jtt_resource:
                    _, identification = train_mlp(xl, yl, transformed["val"], val["y"], vg,
                        config, seed, epochs=config.jtt_identification_epochs)
                    mistakes = (predict_mlp(identification, xl) >= 0.5) != yl
                    jtt_weights = np.where(mistakes, config.jtt_upweight, 1.0)
                methods = [("ERM-MLP", "none", 0, None, None, {}, 0, 0),
                           ("JTT", "none", 0, None, jtt_weights, {}, jtt_resource.seconds, jtt_resource.peak_mb)]
                for k in config.ks:
                    # ใช้ random k แต่ละค่าเพื่อให้จำนวนกลุ่มตรงกับ ablation ทุก k
                    random_clusters = np.random.default_rng(seed + k).integers(k, size=len(yl))
                    methods.append((f"GDRO-random-k{k}", "random", k,
                                    cross_label_groups(random_clusters, yl), None, {}, 0, 0))
                for (rep_name, k), (clusters, scores, seconds, peak) in current_discoveries.items():
                    selected_clusters = clusters if rep_name == "error" else clusters[labeled]
                    methods.append((f"GDRO-{rep_name}-k{k}", rep_name, k,
                        cross_label_groups(selected_clusters, yl), None, scores, seconds, peak))
                _, oracle_groups = np.unique(reference_groups(train["s"].iloc[labeled])["intersection"], return_inverse=True)
                methods.append(("GDRO-oracle", "oracle", 4, oracle_groups, None, {}, 0, 0))
                for method, rep_name, k, groups, weights, alignment, discovery_seconds, discovery_peak in methods:
                    with ResourceMonitor() as fit_resource:
                        checkpoints, _ = train_mlp(xl, yl, transformed["val"], val["y"], vg,
                                                   config, seed, groups=groups, sample_weights=weights)
                    decision_seconds = fit_resource.seconds + (jtt_resource.seconds if method == "JTT" else 0)
                    # JTT รอบแรกนับใน decision time แล้ว ไม่บวกซ้ำใน discovery
                    actual_discovery = 0 if method == "JTT" else discovery_seconds
                    base = dict(common, method=method, representation=rep_name, k=k, **alignment,
                        decision_train_seconds=decision_seconds, discovery_train_seconds=actual_discovery,
                        train_seconds=pre_resource.seconds + actual_discovery + decision_seconds,
                        peak_memory_mb=max(pre_resource.peak_mb, fit_resource.peak_mb, discovery_peak),
                        n_dro_groups=0 if groups is None else len(np.unique(groups)))
                    for selection, (model, epoch, score) in checkpoints.items():
                        record_decision(rows, model, lambda xx, model=model: predict_mlp(model, xx),
                            (selection, epoch, score), base, dict(test, x=transformed["test"]), config, env, output)
                pd.DataFrame(cluster_rows).to_csv(output / "clustering_results.csv", index=False)
                pd.DataFrame(resource_rows).to_csv(output / "representation_resources.csv", index=False)
            audits.append(audit)
            save_json(output / "leakage_audit.json", audits)
            for resource in resource_rows:
                if resource["seed"] == seed:
                    resource["mlflow_run_id"] = log_row(resource, env["libraries"], config, "representation")
            pd.DataFrame(resource_rows).to_csv(output / "representation_resources.csv", index=False)
        generate_reports(output)
    env.update(wall_seconds=time.perf_counter() - start, peak_process_rss_mb=overall.peak_mb,
               decision_rows=len(rows), clustering_rows=len(cluster_rows),
               completed=True, colab_runtime_verified=False)
    save_json(output / "environment.json", env)
    print(f"DONE: {len(rows)} decision rows, {len(cluster_rows)} clustering rows, {env['wall_seconds']:.2f} seconds", flush=True)
    return env


In [ ]:
%%writefile src/fairgroups/clustering.py
"""จัดกลุ่มและวัดความตรงกับกลุ่มอ้างอิงหลัง fit เท่านั้น"""
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, davies_bouldin_score)
from .data import reference_groups


def discover(representation, ids, sensitive, k, seed):
    """ข้อมูลประชากรใช้คำนวณ ARI/NMI เท่านั้น ไม่ส่งให้ k-means"""
    model = KMeans(n_clusters=k, n_init=10, random_state=seed)
    clusters = model.fit_predict(representation)
    model.fit_ids_ = np.asarray(ids).copy()
    scores = {"silhouette": float(silhouette_score(representation, clusters,
               sample_size=min(1000, len(clusters)), random_state=seed)),
              "davies_bouldin": float(davies_bouldin_score(representation, clusters)),
              "reconstruction_error": np.nan}
    for name, values in reference_groups(sensitive).items():
        scores[f"ari_{name}"] = adjusted_rand_score(values, clusters)
        scores[f"nmi_{name}"] = normalized_mutual_info_score(values, clusters)
    return clusters, scores, model


def cross_label_groups(clusters, y):
    """DRO เห็น cluster × y เฉพาะแถวที่มีป้ายกำกับ"""
    _, contiguous = np.unique(np.column_stack([clusters, y]), axis=0, return_inverse=True)
    return contiguous


In [ ]:
%%writefile src/fairgroups/data.py
"""โหลด ACS และแยกข้อมูลประชากรออกก่อนแปลงตัวแปร"""
from pathlib import Path
import numpy as np
import pandas as pd
from folktables import ACSDataSource, ACSIncome
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SENSITIVE = {"SEX", "RAC1P"}
CATEGORICAL = ["COW", "SCHL", "MAR", "OCCP", "POBP", "RELP"]
NUMERIC = ["AGEP", "WKHP"]
FEATURES = NUMERIC + CATEGORICAL


def load_data(root="data"):
    """ใช้แคชเดิมก่อน ถ้าไม่มีจึงให้ folktables ดาวน์โหลด"""
    cache = Path(root) / "2018/1-Year/psam_p06.csv"
    source = ACSDataSource(survey_year="2018", horizon="1-Year",
                           survey="person", root_dir=str(root))
    frame = source.get_data(states=["CA"], download=not cache.exists())
    # ใช้กติกาคัดคนของ ACSIncome โดยเก็บเลขแถวต้นฉบับไว้ตรวจข้อมูลรั่ว
    filtered = ACSIncome._preprocess(frame)
    x = filtered[FEATURES].fillna(-1).copy()
    y = (filtered["PINCP"].to_numpy() > 50000).astype(np.int64)
    sensitive = filtered[["SEX", "RAC1P"]].astype(int).copy()
    del frame, filtered
    return x, y, sensitive


def reference_groups(sensitive):
    """กลุ่มหลักใช้ White/Non-white เพื่อลดปัญหากลุ่มเล็ก"""
    sex = sensitive.SEX.to_numpy().astype(str)
    race = np.where(sensitive.RAC1P.to_numpy() == 1, "White", "Non-white")
    return {"sex": sex, "race": race,
            "intersection": np.char.add(np.char.add(sex, "_"), race),
            "race_detail": sensitive.RAC1P.to_numpy().astype(str)}


def split_data(x, y, sensitive, seed, max_rows=0):
    """สุ่มนำร่องและแบ่ง 60/20/20 ด้วย y ร่วมกับเพศและเชื้อชาติแบบสองกลุ่ม"""
    positions = np.arange(len(y))
    strata = np.char.add(reference_groups(sensitive)["intersection"], y.astype(str))
    if max_rows and len(y) > max_rows:
        positions, _ = train_test_split(positions, train_size=max_rows,
                                       stratify=strata, random_state=seed)
    train, holdout = train_test_split(positions, test_size=0.4,
                                     stratify=strata[positions], random_state=seed)
    val, test = train_test_split(holdout, test_size=0.5,
                                stratify=strata[holdout], random_state=seed)
    return {name: {"x": x.iloc[pos].copy(), "y": y[pos],
                   "s": sensitive.iloc[pos].copy(), "ids": x.index.to_numpy()[pos]}
            for name, pos in [("train", train), ("val", val), ("test", test)]}


def labeled_indices(y, fraction, seed):
    """เลือกป้ายกำกับด้วย y เท่านั้น ไม่ต้องรู้ข้อมูลประชากร"""
    ids = np.arange(len(y))
    if fraction == 1:
        return ids
    chosen, _ = train_test_split(ids, train_size=fraction, stratify=y,
                                random_state=seed)
    return np.sort(chosen)


class TrainPreprocessor:
    """รวมค่าหายากและ one-hot โดยเรียนรู้จาก train เท่านั้น"""
    def __init__(self, min_frequency=20):
        self.encoder = ColumnTransformer([
            ("numeric", StandardScaler(), NUMERIC),
            ("category", OneHotEncoder(handle_unknown="infrequent_if_exist",
             min_frequency=min_frequency, sparse_output=False, dtype=np.float32),
             CATEGORICAL)], sparse_threshold=0)

    def fit(self, frame):
        """เก็บเลขแถวและชื่อตัวแปรไว้เป็นหลักฐาน"""
        assert not SENSITIVE.intersection(frame.columns)
        self.feature_names_in_ = list(frame.columns)
        self.fit_ids_ = frame.index.to_numpy().copy()
        self.encoder.fit(frame)
        return self

    def transform(self, frame):
        return self.encoder.transform(frame).astype(np.float32)


def categorical_frames(splits):
    """LightGBM ใช้หมวดหมู่จาก train และให้ค่าใหม่เป็น missing"""
    categories = {c: sorted(splits["train"]["x"][c].unique()) for c in CATEGORICAL}
    result = {}
    for name, part in splits.items():
        frame = part["x"].copy()
        for col in CATEGORICAL:
            frame[col] = pd.Categorical(frame[col], categories=categories[col])
        result[name] = frame
    return result


In [ ]:
%%writefile src/fairgroups/demo.py
"""Gradio สำหรับวิจัย: ค่าเริ่มต้นจาก train และไม่ต้องรับข้อมูลประชากร"""
import os
import sys
if sys.platform=='darwin': os.environ['OMP_NUM_THREADS']='1'
os.environ.setdefault('GRADIO_ANALYTICS_ENABLED','False')
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from .models import predict_mlp
from .representations import encode_scarf
from .explanations import card_text

PRIMARY=['loan_amount','income','property_value','loan_to_value_ratio','debt_to_income_ratio','loan_term','loan_type','lien_status',
         'tract_to_msa_income_percentage','tract_population','total_units']


def save_bundle(output,bundle):
    cards=pd.read_csv(output/'group_explanations.csv')
    cards=cards[(cards.seed==bundle['seed']) & (cards.method==bundle['method'])]
    if 'group_definition' in cards:
        cards=cards[cards.group_definition.eq(bundle['group_definition'])]
    bundle['cards']={int(r['group']):card_text(r) for r in cards.to_dict('records')}
    bundle['inputs']=[c for c in PRIMARY if c in bundle['defaults']][:8]
    joblib.dump(bundle,output/'demo_bundle.joblib',compress=3)


def load_bundle(output='results'):
    # โหลดเฉพาะไฟล์ที่สร้างเองในโฟลเดอร์งาน
    return joblib.load(Path(output)/'demo_bundle.joblib')


def predict_application(bundle,*values):
    if len(values)!=len(bundle['inputs']): raise ValueError('จำนวนตัวแปรไม่ตรงหน้าสาธิต')
    application=dict(bundle['defaults']);application.update(dict(zip(bundle['inputs'],values)))
    for c in bundle['processor'].categorical_: application[c]=str(application[c])
    frame=bundle['guard'].transform(pd.DataFrame([application]))
    x=bundle['processor'].transform(frame)
    erm=float(predict_mlp(bundle['erm'],x)[0]); fair=float(predict_mlp(bundle['fair'],x)[0])
    rep=x if bundle['representation']=='raw' else bundle['scarf_scaler'].transform(encode_scarf(bundle['scarf'],x)).astype(np.float32)
    group=int(bundle['clusterer'].predict(rep)[0])
    result={'โอกาสอนุมัติ ERM':erm,'โอกาสอนุมัติหลังฝึก DRO':fair,'กลุ่ม':group,'วิธี':bundle['method']}
    result['รูปแบบ DRO']=bundle.get('dro_variant','cy')
    if bundle.get('postprocess'):
        pp=bundle['postprocess']
        pp_rep=x if pp['representation']=='raw' else bundle['scarf_scaler'].transform(encode_scarf(bundle['scarf'],x)).astype(np.float32)
        pp_group=pp['clusterer'].predict(pp_rep)
        score=pp['base_model'].predict_proba(x,num_iteration=pp['iteration'])[:,1]
        threshold=pp['threshold_model'].thresholds_.get(pp_group[0],.5)
        result.update({'ฐานปรับเกณฑ์':pp['base'],'คะแนนเดิมก่อนปรับเกณฑ์':float(score[0]),
            'กลุ่มปรับเกณฑ์':int(pp_group[0]),'เกณฑ์รายกลุ่ม':float(threshold),
            'อนุมัติหลังปรับเกณฑ์':bool(pp['threshold_model'].predict(score,pp_group)[0]),
            'วิธีปรับเกณฑ์':pp['method'],'เป้าหมายปรับเกณฑ์':pp['objective']})
    warning='⚠️ ต้นแบบเพื่อการวิจัยเท่านั้น ความน่าจะเป็นยังไม่ผ่านการปรับเทียบ ไม่ใช้อนุมัติสินเชื่อจริง และผลทดลองอาจไม่ได้ลดอคติ'
    return result,bundle['cards'].get(group,'ยังไม่มีการ์ดกลุ่มนี้')+'\n\n'+warning


def build_interface(output='results'):
    import gradio as gr
    bundle=load_bundle(output)
    inputs=[]
    for c in bundle['inputs']:
        default=bundle['defaults'][c]
        inputs.append(gr.Number(label=c,value=float(default)) if c in bundle['processor'].numeric_ else
                      gr.Textbox(label=c,value=str(default)))
    return gr.Interface(fn=lambda *values:predict_application(bundle,*values),inputs=inputs,
                        outputs=[gr.JSON(label='ผลคาดการณ์'),gr.Markdown(label='คำอธิบายกลุ่ม')],
                        title='ต้นแบบ HMDA: ค้นหากลุ่มผู้ขอสินเชื่อ',
                        description='เพื่อการวิจัยเท่านั้น ใช้ข้อมูลหลักประมาณ 8 ตัว; ช่องอื่นใช้ค่ากลาง/ฐานนิยมจาก train ไม่ใช่ระบบอนุมัติจริง',
                        flagging_mode='never',analytics_enabled=False)


if __name__=='__main__':
    build_interface().launch(share=True)


In [ ]:
%%writefile src/fairgroups/dro.py
"""อัปเดตน้ำหนักกลุ่มแบบ exponentiated gradient ของ group DRO"""
import numpy as np
import torch


def balanced_cluster_weights(clusters, y):
    """w_i=n_c/(จำนวนคลาสที่มีใน c × n_cy); แต่ละคลาสมีมวลเท่ากันภายใน cluster"""
    weights = np.ones(len(y), dtype=np.float32)
    for cluster in np.unique(clusters):
        mask = clusters == cluster
        labels, counts = np.unique(y[mask], return_counts=True)
        for label, count in zip(labels, counts):
            weights[mask & (y == label)] = mask.sum() / (len(labels) * count)
    return weights


def validation_groups(clusters, y=None, definition='c'):
    """c ใช้ cluster อย่างเดียว; cy เพิ่ม y โดย Unknown ยังคงถูกตัดออกสำหรับ oracle"""
    if definition == 'c':
        return np.asarray(clusters).copy()
    if definition != 'cy' or y is None:
        raise ValueError('cy ต้องมี y และ definition ต้องเป็น c หรือ cy')
    return np.array(['Unknown' if str(c) == 'Unknown' else f'{c}|y={label}'
                     for c, label in zip(clusters, y)])


def make_dro_groups(x, clusters, min_size=50, definition='c', y=None):
    """รวมกลุ่ม c โดยไม่ดู y; cy รวมได้เฉพาะคลาสเดียวกันตามรอบก่อน"""
    if definition not in ('c', 'cy') or (definition == 'cy' and y is None):
        raise ValueError('นิยามกลุ่มไม่ถูกต้อง')
    labels = np.zeros(len(clusters), dtype=int) if definition == 'c' else y
    groups, details = merge_small_groups(x, labels, clusters, min_size)
    if definition == 'c':
        for row in details['mapping']:
            row.pop('y')
    details['group_definition'] = definition
    return groups, details


def merge_small_groups(x, y, clusters, min_size=50):
    """รวม cluster × y ที่เล็กด้วยระยะศูนย์กลาง train ในคลาสเดียวกันเท่านั้น"""
    pairs, groups = np.unique(np.column_stack([clusters, y]), axis=0, return_inverse=True)
    initial = groups.copy()
    active = {g: {'label': pair[1], 'n': int((groups == g).sum()),
                  'center': x[groups == g].mean(axis=0)} for g, pair in enumerate(pairs)}
    while True:
        small = [g for g, d in active.items() if d['n'] < min_size and
                 any(h != g and e['label'] == d['label'] for h, e in active.items())]
        if not small:
            break
        source = min(small, key=lambda g: (active[g]['n'], g))
        a = active[source]
        same_class = [g for g, d in active.items() if g != source and d['label'] == a['label']]
        target = min(same_class, key=lambda g: (float(np.sum((active[g]['center'] - a['center'])**2)), g))
        b = active[target]
        b['center'] = (b['center'] * b['n'] + a['center'] * a['n']) / (b['n'] + a['n'])
        b['n'] += a['n']
        groups[groups == source] = target
        del active[source]
    _, merged = np.unique(groups, return_inverse=True)
    details = {'n_groups_before': len(pairs), 'n_groups_after': len(active),
               'min_size': min_size, 'fit_split': 'train',
               'mapping': [{'cluster': int(c), 'y': int(label),
                            'dro_group': int(merged[np.flatnonzero(initial == g)[0]])}
                           for g, (c, label) in enumerate(pairs)],
               'group_sizes': np.bincount(merged).tolist()}
    # ถ้าทั้งคลาสมีน้อยกว่าเกณฑ์ จะเหลือหนึ่งกลุ่มและรายงาน ไม่รวมข้ามคลาส
    details['undersized_remaining'] = sum(n < min_size for n in details['group_sizes'])
    return merged, details


def group_dro_loss(losses, groups, log_weights, eta, group_counts=None, adjustment=0.0):
    """เฉลี่ย loss ในแต่ละกลุ่มก่อนถ่วงน้ำหนัก กลุ่มว่างไม่ทำให้ NaN"""
    means = torch.zeros_like(log_weights)
    present = torch.zeros_like(log_weights, dtype=torch.bool)
    for group in groups.unique():
        present[group] = True
        means[group] = losses[groups == group].mean()
    with torch.no_grad():
        adjusted = means.detach().clone()
        if adjustment:
            if group_counts is None:
                raise ValueError('ต้องใช้ขนาดกลุ่มจาก train ทั้งชุดสำหรับ C/sqrt(n_g)')
            adjusted[present] += adjustment / group_counts[present].sqrt()
        # กลุ่มที่ไม่อยู่ใน batch ไม่ถูกเพิ่ม loss สมมติ
        log_weights.add_(eta * adjusted)
        log_weights.sub_(torch.logsumexp(log_weights, dim=0))
    weights = torch.softmax(log_weights, dim=0) * present
    weights = weights / weights.sum()
    return (weights * means).sum()


In [ ]:
%%writefile src/fairgroups/efficiency.py
"""วัดทรัพยากรจริงและนับต้นทุนของชั้น Linear"""
import platform
import threading
import time
import importlib.metadata as metadata
import subprocess
from pathlib import Path
import psutil
import torch


class ResourceMonitor:
    """สุ่ม RSS ทุก 10 ms ค่านี้รวมหน่วยความจำของกระบวนการ Python"""
    def __enter__(self):
        self.process = psutil.Process()
        self.peak = self.process.memory_info().rss
        self.peak_start = self.peak
        self.start = time.perf_counter()
        self.stop = threading.Event()
        self.worker = threading.Thread(target=self._sample, daemon=True)
        self.worker.start()
        return self

    def _sample(self):
        while not self.stop.wait(0.01):
            self.peak = max(self.peak, self.process.memory_info().rss)

    def __exit__(self, *args):
        self.stop.set()
        self.worker.join()
        self.peak = max(self.peak, self.process.memory_info().rss)
        self.seconds = time.perf_counter() - self.start
        self.peak_mb = self.peak / 1024**2


def network_cost(model):
    """FLOPs นับเฉพาะ Linear ตามโจทย์ ไม่รวม activation/bias"""
    return {"parameters": sum(p.numel() for p in model.parameters()),
            "flops_per_row": sum(2 * m.in_features * m.out_features
                                 for m in model.modules() if isinstance(m, torch.nn.Linear))}


def inference_seconds(predict, x):
    """อุ่นหนึ่งรอบแล้วเฉลี่ยสามรอบ คืนวินาทีต่อ 1,000 แถว"""
    sample = x[:min(1000, len(x))]
    predict(sample)
    start = time.perf_counter()
    for _ in range(3):
        predict(sample)
    return (time.perf_counter() - start) / 3 * 1000 / len(sample)


def environment(config):
    """บันทึกเวอร์ชันและทรัพยากรที่มีให้ใช้"""
    packages = ["numpy", "pandas", "scipy", "threadpoolctl", "scikit-learn", "lightgbm", "torch",
                "mlflow", "folktables", "matplotlib", "psutil", "pytest", "gradio", "unsupervised-bias-detection", "joblib"]
    cpu_model = platform.processor() or platform.machine()
    if platform.system() == "Darwin":
        try:
            cpu_model = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        except (OSError, subprocess.CalledProcessError):
            pass
    elif Path("/proc/cpuinfo").exists():
        cpu_model = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                          if line.startswith("model name")), cpu_model)
    return {"python": platform.python_version(), "platform": platform.platform(),
            "libraries": {p: metadata.version(p) for p in packages},
            "installed_packages": dict(sorted((d.metadata["Name"], d.version) for d in metadata.distributions() if d.metadata["Name"])),
            "cpu": cpu_model,
            "logical_cpus": psutil.cpu_count(),
            "ram_gb": psutil.virtual_memory().total / 1024**3,
            "gpu_available": torch.cuda.is_available(), "device_used": "cpu",
            "config": config.to_dict(), "seeds": list(config.seeds)}


In [ ]:
%%writefile src/fairgroups/explanations.py
"""กฎจากตัวแปรเดิม และทดสอบความต่างบน test ที่ไม่ใช้สร้างกลุ่ม"""
import json
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import f1_score, precision_score, recall_score
from .hmda import HMDAPreprocessor, NUMERIC, reference_groups


def fit_surrogate(x, target, seed, class_weight="balanced"):
    """ถ่วงสมดุลเพื่อให้กลุ่มเล็กมีโอกาสถูกอธิบาย และรักษาความลึกไม่เกิน 3"""
    n_group = int(target.sum()); n_rest = len(target) - n_group
    tree = DecisionTreeClassifier(max_depth=3, class_weight=class_weight,
        min_samples_leaf=max(5, min(n_group, n_rest) // 20), random_state=seed)
    return tree.fit(x, target)


def choose_surrogate(x, target, xv, val_target, seed):
    """เลือก balanced หรือไม่ถ่วงด้วย F1 validation ของ cluster เท่านั้น; เสมอเลือก balanced"""
    candidates = [(name, fit_surrogate(x, target, seed, weight))
                  for name, weight in [('balanced', 'balanced'), ('unbalanced', None)]]
    scores = {name: float(f1_score(val_target, tree.predict(xv), zero_division=0))
              for name, tree in candidates}
    name, tree = max(candidates, key=lambda item: scores[item[0]])
    return tree, {'surrogate_class_weight': name, 'fidelity_f1_balanced': scores['balanced'],
                  'fidelity_f1_unbalanced': scores['unbalanced'], 'surrogate_selection_split': 'validation'}


def feature_differences(frame, mask, limit=5):
    """คำอธิบายสำรองจาก train: SMD แบบ signed หรือส่วนต่างสัดส่วนที่ต่างที่สุดต่อคอลัมน์"""
    differences = []
    for feature in frame:
        a, b = frame.loc[mask, feature], frame.loc[~mask, feature]
        if feature in NUMERIC:
            a = pd.to_numeric(a, errors='coerce').dropna()
            b = pd.to_numeric(b, errors='coerce').dropna()
            if len(a) < 2 or len(b) < 2:
                continue
            delta = float(a.mean() - b.mean())
            scale = float(np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2))
            if scale == 0:
                if delta == 0:
                    continue
                score = np.inf
                value = None
            else:
                value = delta / scale; score = abs(value)
            row = {'feature': feature, 'metric': 'standardized_mean_difference',
                   'value': value, 'zero_within_group_variance': scale == 0,
                   'group_mean': float(a.mean()), 'rest_mean': float(b.mean())}
        else:
            # นับ missing เป็นหนึ่งหมวด เพื่อให้สัดส่วนรวมเป็นหนึ่งในแต่ละฝั่ง
            a = a.astype('string').fillna('<missing>').value_counts(normalize=True)
            b = b.astype('string').fillna('<missing>').value_counts(normalize=True)
            delta = a.reindex(a.index.union(b.index), fill_value=0) - b.reindex(a.index.union(b.index), fill_value=0)
            if delta.empty:
                continue
            category = delta.abs().idxmax(); value = float(delta[category]); score = abs(value)
            row = {'feature': feature, 'category': str(category), 'metric': 'proportion_difference',
                   'value': value, 'group_proportion': float(a.get(category, 0)),
                   'rest_proportion': float(b.get(category, 0))}
        differences.append((score, row))
    # เป็น effect size เพื่อพรรณนา ไม่ใช่ p-value และไม่ใช้เลือกวิธีฝึก
    return [row for _, row in sorted(differences, key=lambda item: item[0], reverse=True)[:limit]]


def rules(tree, names):
    """แสดงเฉพาะเส้นทางใบที่ทายว่าอยู่กลุ่มนี้ แต่ละเส้นทางไม่เกิน 3 เงื่อนไข"""
    found=[]
    def visit(node, conditions):
        if tree.tree_.feature[node] == _tree.TREE_UNDEFINED:
            if tree.classes_[np.argmax(tree.tree_.value[node][0])] == 1:
                found.append(('IF ' + ' AND '.join(conditions) if conditions else 'IF TRUE') + ' THEN อยู่กลุ่มนี้')
            return
        feature=names[tree.tree_.feature[node]].replace('numeric__','').replace('category__','')
        threshold=tree.tree_.threshold[node]
        visit(tree.tree_.children_left[node], conditions+[f'{feature} ≤ {threshold:.4g}'])
        visit(tree.tree_.children_right[node], conditions+[f'{feature} > {threshold:.4g}'])
    visit(0,[])
    return found or ['ไม่มีใบที่ทำนายว่าอยู่กลุ่มนี้ (อ่าน F1 ควบคู่)']


def describe_groups(method, k, seed, train, val, test, labels, erm_probability, config):
    processor=HMDAPreprocessor(config.min_category_frequency,config.lei_top_n,scale=False).fit(train['x'])
    xtr,xv = processor.transform(train['x']),processor.transform(val['x'])
    rows, tests=[] , []
    for group in np.unique(labels['train']):
        target=(labels['train']==group).astype(int)
        vt=(labels['val']==group).astype(int)
        tree, choice=choose_surrogate(xtr,target,xv,vt,seed)
        vp=tree.predict(xv)
        mask=labels['test']==group; yy=test['y'][mask]; pp=(erm_probability[mask]>=.5).astype(int)
        n=int(mask.sum()); positive=yy==1; negative=yy==0
        population={}
        for dimension, values in reference_groups(test['s']).items():
            if dimension=='intersection': continue
            population[dimension]={str(v):{'group':float((values[mask]==v).mean()) if n else None,
                                          'overall':float((values==v).mean())} for v in np.unique(values)}
        paths=rules(tree,processor.names())
        positive_leaves=[]
        def lengths(node,depth=0):
            if tree.tree_.feature[node]==_tree.TREE_UNDEFINED:
                if tree.classes_[np.argmax(tree.tree_.value[node][0])]==1: positive_leaves.append(depth)
            else:
                lengths(tree.tree_.children_left[node],depth+1); lengths(tree.tree_.children_right[node],depth+1)
        lengths(0)
        row={'seed':seed,'method':method,'k':k,'group':int(group),'n_train':int(target.sum()),'n_val':int(vt.sum()),'n_test':n,
             **choice, 'fidelity_f1':float(f1_score(vt,vp,zero_division=0)),
             'fidelity_precision':float(precision_score(vt,vp,zero_division=0)),
             'fidelity_recall':float(recall_score(vt,vp,zero_division=0)),
             'fallback_features':json.dumps(feature_differences(train['x'],target.astype(bool)),ensure_ascii=False),
             'fallback_split':'train', 'rule_length':sum(positive_leaves),
             'max_conditions_per_path':max(positive_leaves,default=0),'n_rules':len(positive_leaves),
             'coverage':float(vp.mean()),'actual_coverage_test':float(mask.mean()),
             'approval_rate':float(yy.mean()) if n else np.nan,'erm_error':float((yy!=pp).mean()) if n else np.nan,
             'fpr':float(pp[negative].mean()) if negative.any() else np.nan,
             'fnr':float((1-pp[positive]).mean()) if positive.any() else np.nan,
             'rules':json.dumps(paths,ensure_ascii=False),'population':json.dumps(population,ensure_ascii=False),
             'rule_fit_split':'train','fidelity_split':'validation','statistical_test_split':'test'}
        rows.append(row)
        for c in test['x']:
            a,b=test['x'].loc[mask,c],test['x'].loc[~mask,c]
            p=np.nan
            if c in NUMERIC:
                a,b=a.dropna(),b.dropna()
                if len(a)>1 and len(b)>1 and (a.var()>0 or b.var()>0):
                    p=float(ttest_ind(a,b,equal_var=False).pvalue)
                test_name='Welch t-test'
            else:
                table=pd.crosstab(pd.Series(mask,index=test['x'].index),test['x'][c])
                if table.shape[0]==2 and table.shape[1]>1:
                    p=float(chi2_contingency(table,correction=False)[1])
                test_name='chi-square'
            tests.append({'seed':seed,'method':method,'k':k,'group':int(group),'feature':c,'test':test_name,'p_value':p})
    return rows,tests,processor.fit_ids_


def card_text(row):
    r=row
    def fmt(v): return f'{v:.4f}' if pd.notna(v) else 'N/A'
    pop=json.loads(r['population']); composition=[]
    for dim, vals in pop.items():
        composition.append(dim+': '+', '.join(f"{v} {d['group']:.1%} (ทั้งชุด {d['overall']:.1%})" for v,d in vals.items() if d['group'] is not None))
    fallback=[]
    if r['fidelity_f1'] < .5:
        fallback.append('- **กลุ่มนี้อธิบายด้วยกฎลึก 3 ชั้นได้ไม่ดี**')
        fallback.append('- คำอธิบายสำรอง: ตัวแปรที่ต่างจากส่วนที่เหลือมากที่สุด คำนวณบน train เท่านั้น')
        for d in json.loads(r.get('fallback_features','[]')):
            if d['metric']=='standardized_mean_difference':
                value = 'ไม่กำหนด (ความแปรปรวนภายในเป็นศูนย์)' if d['value'] is None else f"{d['value']:.3f}"
                fallback.append(f"  - {d['feature']}: SMD {value}; ค่าเฉลี่ยกลุ่ม {d['group_mean']:.4g}, ส่วนที่เหลือ {d['rest_mean']:.4g}")
            else:
                fallback.append(f"  - {d['feature']} = {d['category']}: สัดส่วนต่าง {d['value']:+.3f}; กลุ่ม {d['group_proportion']:.1%}, ส่วนที่เหลือ {d['rest_proportion']:.1%}")
    return '\n'.join([f"### {r['method']} · {r.get('group_definition', 'cy')} · seed {r['seed']} · กลุ่ม {r['group']}",
        *['- '+v for v in json.loads(r['rules'])],
        f"- ขนาด train/validation/test: {r['n_train']}/{r['n_val']}/{r['n_test']} · coverage กฎบน validation {fmt(r['coverage'])}",
        f"- Fidelity F1 {fmt(r['fidelity_f1'])} · precision {fmt(r.get('fidelity_precision',np.nan))} · recall {fmt(r.get('fidelity_recall',np.nan))} · เงื่อนไขรวม {r['rule_length']} · สูงสุดต่อเส้นทาง {r['max_conditions_per_path']}",
        *([f"- ต้นไม้ที่เลือก: {r['surrogate_class_weight']} ด้วย F1 validation ของ cluster"] if 'surrogate_class_weight' in r else []),
        *fallback,
        f"- อนุมัติจริง {fmt(r['approval_rate'])} · ERM error {fmt(r['erm_error'])} · FPR {fmt(r['fpr'])} · FNR {fmt(r['fnr'])}",
        '- ตัวแปรต่างชัดหลัง Bonferroni: '+r.get('significant_features',''),
        *['- องค์ประกอบ '+v for v in composition],
        '- สถิติและการทดสอบใช้ test เท่านั้น; กฎ fit train และวัด fidelity บน validation', ''])


def export_explanations(output, rows, tests):
    tests=pd.DataFrame(tests)
    # ปรับทั้ง family ของการทดสอบทุก seed/วิธี/กลุ่ม/ตัวแปรที่ส่งออก
    tests['p_bonferroni']=(tests.p_value*tests.p_value.notna().sum()).clip(upper=1)
    tests['significant']=tests.p_bonferroni<.05
    tests.to_csv(output/'group_feature_tests.csv',index=False)
    for row in rows:
        matching=tests[(tests.seed==row['seed']) & (tests.method==row['method']) & (tests.group==row['group']) & tests.significant]
        if 'group_definition' in row:
            matching=matching[matching.group_definition.eq(row['group_definition'])]
        row['significant_features']=', '.join(f"{r.feature} (p_adj={r.p_bonferroni:.2g})" for r in matching.itertuples()) or 'ไม่พบ'
    frame=pd.DataFrame(rows);frame.to_csv(output/'group_explanations.csv',index=False)
    (output/'group_cards.md').write_text('# การ์ดอธิบายกลุ่ม HMDA\n\nกฎ one-hot ที่ ≤ 0.5 หมายถึงไม่มีค่านั้น; สถิติต่างกลุ่มเป็นเชิงสำรวจ ไม่ใช่เหตุและผล\n\n'+'\n'.join(card_text(r) for r in rows),encoding='utf-8')
    return rows


In [ ]:
%%writefile src/fairgroups/hmda.py
"""ข้อมูล HMDA: แยกประชากรและตัวแปรหลังตัดสินใจก่อนฝึกทุกขั้น"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

POST = set('action_taken purchaser_type interest_rate rate_spread hoepa_status total_loan_costs total_points_and_fees origination_charges discount_points lender_credits prepayment_penalty_term intro_rate_period negative_amortization interest_only_payment balloon_payment other_nonamortizing_features multifamily_affordable_units activity_year state_code'.split())
NUMERIC = 'loan_amount loan_to_value_ratio property_value income debt_to_income_ratio loan_term total_units tract_population ffiec_msa_md_median_family_income tract_to_msa_income_percentage tract_owner_occupied_units tract_one_to_four_family_homes tract_median_age_of_housing_units'.split()
CATEGORICAL = 'loan_type lien_status preapproval construction_method applicant_credit_score_type submission_of_application initially_payable_to_institution aus-1 conforming_loan_limit derived_loan_product_type derived_msa-md county_code lei'.split()
FEATURES = NUMERIC + CATEGORICAL
UNKNOWN = 'Unknown'


def sensitive_column(c):
    """credit_score_type ไม่ใช่คะแนนเครดิตจริง และไม่ใช่ตัวแปรประชากร"""
    return c in {'derived_race', 'derived_ethnicity', 'derived_sex'} or (
        c.startswith(('applicant_', 'co-applicant_')) and
        any(t in c for t in ('race', 'ethnicity', 'sex', 'age', 'observed')))


def prohibited(c):
    return c in POST or c.startswith('denial_reason-') or sensitive_column(c) or c == 'tract_minority_population_percent'


def missing(series):
    return series.isna() | series.astype(str).str.strip().str.lower().isin(['', 'exempt', 'na', 'n/a', 'nan', 'none'])


def dti_numeric(series):
    """แปลงช่วงเป็นค่ากึ่งกลางที่มีลำดับ ไม่เดาค่าจากผลอนุมัติ"""
    mapping = {'<20%': 10., '20%-<30%': 25., '30%-<36%': 33., '50%-60%': 55., '>60%': 65.}
    return pd.to_numeric(series.replace(mapping), errors='coerce')


def demographics(frame):
    race, eth = frame.derived_race.fillna(UNKNOWN), frame.derived_ethnicity.fillna(UNKNOWN)
    unavailable = race.str.contains('Not Available|Free Form Text Only|Unknown', case=False) | eth.str.contains('Not Available|Free Form Text Only|Unknown', case=False)
    # Hispanic รวมทุกเชื้อชาติ แม้ไม่มี race; ไม่มี ethnicity ไม่สมมุติว่า NH
    re = np.where(eth == 'Hispanic or Latino', 'Hispanic', np.where(unavailable, UNKNOWN,
         race.map({'White':'NH White', 'Black or African American':'NH Black', 'Asian':'NH Asian'}).fillna('Other')))
    sex = frame.derived_sex.where(frame.derived_sex.isin(['Male', 'Female', 'Joint']), UNKNOWN)
    age = frame.applicant_age.astype(str).where(frame.applicant_age.astype(str).isin(['<25','25-34','35-44','45-54','55-64','65-74','>74']), UNKNOWN)
    result = pd.DataFrame({'race_eth': re, 'sex': sex, 'age': age}, index=frame.index)
    result['intersection'] = np.where((result.race_eth != UNKNOWN) & (result.sex != UNKNOWN), result.race_eth + ' × ' + result.sex, UNKNOWN)
    return result


def reference_groups(s):
    return {c: s[c].to_numpy().astype(str) for c in ['race_eth','sex','age','intersection']}


def load_hmda(root, state='GA'):
    path = Path(root) / 'hmda' / f'hmda_2023_{state}.csv'
    if not path.exists():
        # urllib ติดตาม redirect ตามมาตรฐาน และเก็บแคชใช้รอบถัดไป
        import urllib.request
        path.parent.mkdir(parents=True, exist_ok=True)
        url = f'https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states={state}&years=2023&actions_taken=1,2,3'
        temp = path.with_suffix('.part')
        urllib.request.urlretrieve(url, temp)
        temp.replace(path)
    frame = pd.read_csv(path, low_memory=False)
    stages = [{'stage':'ข้อมูลต้นทาง', 'n':len(frame)}]
    filters = [('ซื้อที่อยู่อาศัย', pd.to_numeric(frame.loan_purpose, errors='coerce') == 1),
        ('อยู่เอง', frame.occupancy_type == 1), ('ไม่ใช่ธุรกิจ', frame.business_or_commercial_purpose != 1),
        ('ไม่ใช่ reverse mortgage', frame.reverse_mortgage != 1), ('ไม่ใช่ open-end', frame['open-end_line_of_credit'] != 1),
        ('1–4 หน่วย', pd.to_numeric(frame.total_units, errors='coerce').between(1,4)),
        ('Single Family', frame.derived_dwelling_category.str.contains('Single Family', na=False)),
        ('ผลอนุมัติ/ปฏิเสธที่กำหนด', frame.action_taken.isin([1,2,3]))]
    keep = np.ones(len(frame), dtype=bool)
    for label, mask in filters:
        keep &= mask.to_numpy()
        stages.append({'stage':label,'n':int(keep.sum())})
    frame = frame.loc[keep].copy()
    y = frame.action_taken.isin([1,2]).to_numpy().astype(int)
    return frame, y, demographics(frame), pd.DataFrame(stages)


def split_hmda(frame, y, s, seed, max_rows=0):
    positions = np.arange(len(y)); strata = s.race_eth.to_numpy() + y.astype(str)
    if max_rows and len(y) > max_rows:
        positions, _ = train_test_split(positions, train_size=max_rows, stratify=strata, random_state=seed)
    train, rest = train_test_split(positions, test_size=.4, stratify=strata[positions], random_state=seed)
    val, test = train_test_split(rest, test_size=.5, stratify=strata[rest], random_state=seed)
    return {name: {'frame':frame.iloc[idx].copy(), 'y':y[idx], 's':s.iloc[idx].copy(), 'ids':frame.index.to_numpy()[idx]}
            for name, idx in [('train',train),('val',val),('test',test)]}


class LeakageGuard:
    """เลือกคอลัมน์จาก train เท่านั้น ไม่ใช้ missing ของ validation/test"""
    def fit(self, frame, y):
        self.fit_ids_ = frame.index.to_numpy().copy()
        self.audit_, self.features_ = [], []
        for c in frame.columns:
            m = missing(frame[c]).to_numpy()
            rates = [float(m[y == v].mean()) for v in [0,1]]
            gap = abs(rates[1]-rates[0])
            reason = ('ข้อมูลประชากร/หลังตัดสินใจ/ตัวแทนพื้นที่ที่ห้ามใช้' if prohibited(c) else
                      'missing gap > 10 จุดเปอร์เซ็นต์' if gap > .10 else
                      'อยู่นอกรายการตัวแปรก่อนตัดสินใจที่อนุญาต' if c not in FEATURES else '')
            if not reason: self.features_.append(c)
            self.audit_.append({'column':c,'missing_denied':rates[0],'missing_approved':rates[1],
                                'missing_gap':gap,'dropped':bool(reason),'reason':reason})
        assert all(not prohibited(c) for c in self.features_)
        return self

    def transform(self, frame):
        result = frame[self.features_].copy()
        for c in result:
            if c in NUMERIC:
                result[c] = dti_numeric(result[c]) if c == 'debt_to_income_ratio' else pd.to_numeric(result[c], errors='coerce')
                result[c] = result[c].replace([np.inf,-np.inf],np.nan).astype(float)
            else:
                result[c] = result[c].where(~missing(result[c]), '__missing__').astype(str)
        return result


class HMDAPreprocessor:
    """แทนค่าว่าง รวมค่าหายาก และ top-N ผู้ให้กู้ด้วย train เท่านั้น"""
    def __init__(self, min_frequency=20, lei_top_n=30, scale=True):
        self.min_frequency, self.lei_top_n, self.scale = min_frequency, lei_top_n, scale

    def prepare(self, frame):
        frame = frame.copy()
        if 'lei' in frame:
            frame['lei'] = frame.lei.where(frame.lei.isin(self.top_lei_), '__other__')
        return frame

    def fit(self, frame):
        assert all(not prohibited(c) for c in frame.columns)
        self.fit_ids_ = frame.index.to_numpy().copy()
        self.feature_names_in_ = list(frame.columns)
        self.top_lei_ = frame.lei.value_counts().head(self.lei_top_n).index.tolist() if 'lei' in frame else []
        self.numeric_ = [c for c in frame if c in NUMERIC]
        self.categorical_ = [c for c in frame if c not in NUMERIC]
        numeric = [SimpleImputer(strategy='median', keep_empty_features=True)]
        if self.scale: numeric.append(StandardScaler())
        self.encoder = ColumnTransformer([
            ('numeric',make_pipeline(*numeric),self.numeric_),
            ('category',OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=self.min_frequency,
                                     sparse_output=False, dtype=np.float32),self.categorical_)], sparse_threshold=0)
        self.encoder.fit(self.prepare(frame))
        return self

    def transform(self, frame):
        return self.encoder.transform(self.prepare(frame)).astype(np.float32)

    def names(self):
        return self.encoder.get_feature_names_out().tolist()


def write_profile(output, stages, y, s):
    from .reporting import markdown_table
    stages.to_csv(output/'filter_counts.csv',index=False)
    rows=[]
    for name, values in reference_groups(s).items():
        for group in np.unique(values):
            mask=values==group
            rows.append({'dimension':name,'group':group,'n':int(mask.sum()),'approval_rate':float(y[mask].mean()),
                         'denial_rate':float(1-y[mask].mean()),'excluded_fairness':group==UNKNOWN})
    profile=pd.DataFrame(rows); profile.to_csv(output/'data_profile.csv',index=False)
    text=['# ข้อมูล HMDA หลังคัดกรอง','',markdown_table(stages,['stage'],['n']),'',
          'Unknown คงไว้ฝึก แต่ตัดจาก fairness เฉพาะมิติที่ไม่ทราบ และตัดกลุ่มเล็กตาม min_fair_group ใน CSV ผลทดลอง',
          '',markdown_table(profile,['dimension','group'],['n','approval_rate','denial_rate'])]
    (output/'data_profile.md').write_text('\n'.join(text),encoding='utf-8')


In [ ]:
%%writefile src/fairgroups/hmda_groups.py
"""จัดกลุ่ม train ภายในคลาส และทำนายรายใหม่โดยไม่รู้ y"""
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from .hmda import reference_groups, UNKNOWN
from .models import predict_mlp
import torch


class GlobalClusters:
    """จัด k กลุ่มบน representation ทั้งหมดโดยไม่รับ y (สำหรับ c/c_bal)"""
    def fit(self, x, ids, k, seed):
        self.model_ = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(x)
        self.labels_ = self.model_.labels_
        self.centers_ = self.model_.cluster_centers_.astype(np.float32)
        self.fit_ids_ = np.asarray(ids).copy()
        return self

    def predict(self, x):
        return self.model_.predict(x)


class ClassClusters:
    """k ต่อคลาส รวม 2k กลุ่ม; ตอนใช้จริงเลือกศูนย์กลางที่ใกล้สุดข้ามคลาส"""
    def fit(self, x, y, ids, k, seed):
        self.models_, self.centers_, self.labels_ = [], [], np.zeros(len(y), dtype=int)
        for label in [0,1]:
            mask = y == label
            km = KMeans(n_clusters=k,n_init=10,random_state=seed).fit(x[mask])
            self.models_.append(km); self.centers_.append(km.cluster_centers_)
            self.labels_[mask] = label*k + km.labels_
        self.centers_ = np.vstack(self.centers_).astype(np.float32)
        self.fit_ids_ = np.asarray(ids).copy()
        return self

    def predict(self, x):
        from sklearn.metrics import pairwise_distances_argmin
        return pairwise_distances_argmin(x, self.centers_)


class GeoGroups:
    """ใช้ minority percent เฉพาะ baseline นี้ ขอบ quantile มาจาก train"""
    def fit(self, values, ids, k=4):
        self.median_ = float(np.nanmedian(values))
        self.edges_ = np.unique(np.quantile(np.nan_to_num(values,nan=self.median_), np.arange(1,k)/k))
        self.fit_ids_ = np.asarray(ids).copy()
        return self
    def predict(self, values):
        return np.digitize(np.nan_to_num(values,nan=self.median_),self.edges_)


def hidden(model,x):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model.layers[:-1](torch.as_tensor(x[i:i+2048])).numpy() for i in range(0,len(x),2048)])


def group_scores(x, labels, s, seed):
    out = {'n_clusters':len(np.unique(labels)), 'silhouette':np.nan, 'davies_bouldin':np.nan}
    if 1 < len(np.unique(labels)) < len(labels):
        out.update(silhouette=float(silhouette_score(x,labels,sample_size=min(1000,len(labels)),random_state=seed)),
                   davies_bouldin=float(davies_bouldin_score(x,labels)))
    for name, group in reference_groups(s).items():
        keep = group != UNKNOWN
        out[f'ari_{name}'] = adjusted_rand_score(group[keep],labels[keep])
        out[f'nmi_{name}'] = normalized_mutual_info_score(group[keep],labels[keep])
        out[f'alignment_excluded_{name}'] = int((~keep).sum())
    return out


def fit_hbac(x, error, ids, seed, k=4):
    """ใช้ API จริงของแพ็กเกจ; error สูงคือคุณภาพต่ำ จึงส่ง -error"""
    from unsupervised_bias_detection.cluster import BiasAwareHierarchicalKMeans
    model = BiasAwareHierarchicalKMeans(bahc_max_iter=k-1, bahc_min_cluster_size=max(20,len(x)//200),
                                       random_state=seed,n_init=5)
    model.fit(x, -error)
    model.fit_ids_ = np.asarray(ids).copy()
    return model


In [ ]:
%%writefile src/fairgroups/hmda_reporting.py
"""สร้างตารางและกราฟ ablation จาก CSV เท่านั้น ไม่ใช้ test เลือกวิธี"""
import os
from pathlib import Path
import pandas as pd
from .reporting import markdown_table

METRICS=['accuracy','positive_rate','approval_rate_gap_race_eth','approval_rate_gap_sex',
         'eo_gap','worst_group_accuracy']


def main_rows(results):
    """ERM ใช้ average; JTT และ DRO ใช้ inferred_worst_group"""
    erm=results.method.isin(['ERM-LGBM','ERM-MLP'])
    return results[(erm & results.selection.eq('average')) |
                   (~erm & results.selection.eq('inferred_worst_group'))].copy()


def oracle_sanity(results):
    """แยกตาม variant; ตรวจหลังเลือกแล้วและคงผลล้มเหลวไว้"""
    main=main_rows(results)
    erm=main[main.method.eq('ERM-MLP')].set_index('seed').worst_group_accuracy
    rows=[]
    oracle=main[main.method.eq('GDRO-oracle')]
    if 'dro_variant' not in oracle: oracle=oracle.assign(dro_variant='cy')
    for variant,part in oracle.groupby('dro_variant',sort=False):
        assert set(erm.index)==set(part.seed)
        for r in part.itertuples():
            rows.append({'dro_variant':variant,'seed':r.seed,'erm_worst_group':erm.loc[r.seed],
                         'oracle_worst_group':r.worst_group_accuracy,'gain':r.worst_group_accuracy-erm.loc[r.seed]})
    sanity=pd.DataFrame(rows)
    status=sanity.groupby('dro_variant').gain.mean().ge(-1e-12).to_dict()
    gains=sanity.groupby('dro_variant').gain.mean().to_dict()
    return sanity,status,gains


def tuning_tables(output):
    """เก็บประวัติรอบเก่าแยกจาก ablation รอบนี้"""
    fidelity=[];comparison=[]
    for name,folder in [(f'round{i}',output/f'round{i}') for i in [2,3,4]]+[('round5',output)]:
        path=folder/'group_explanations.csv'
        if path.exists():
            frame=pd.read_csv(path)
            for method,part in [('ทุกวิธี',frame),*list(frame.groupby('method'))]:
                fidelity.append({'round':name,'method':method,'n_groups':len(part),
                    'median_fidelity_f1':part.fidelity_f1.median(),'fraction_f1_ge_080':part.fidelity_f1.ge(.8).mean(),
                    'n_f1_below_050':int(part.fidelity_f1.lt(.5).sum())})
        path=folder/'hmda_pilot_results.csv'
        if path.exists():
            frame=pd.read_csv(path)
            main=main_rows(frame) if name in ['round4','round5'] else frame[frame.selection.eq('average')].copy()
            if 'dro_variant' not in main: main['dro_variant']='legacy'
            proposed=main[main.proposed_selected].copy();proposed['method']='Proposed (Silhouette)'
            main=pd.concat([main,proposed],ignore_index=True);main['round']=name
            comparison.append(main[['round','seed','method','dro_variant',*METRICS]])
    fidelity=pd.DataFrame(fidelity);fidelity.to_csv(output/'fidelity_comparison.csv',index=False)
    comparison=pd.concat(comparison,ignore_index=True);comparison.to_csv(output/'round_comparison.csv',index=False)
    return fidelity,comparison


def generate_reports(output):
    output=Path(output)
    results=pd.read_csv(output/'hmda_pilot_results.csv');main=main_rows(results)
    clusters=pd.read_csv(output/'clustering_results.csv');cards=pd.read_csv(output/'group_explanations.csv')
    resources=pd.read_csv(output/'representation_resources.csv');grid=pd.read_csv(output/'dro_grid.csv')
    proposed=main[main.proposed_selected].copy();proposed['method']='Proposed (Silhouette)'
    comparison=pd.concat([main,proposed],ignore_index=True)
    baseline=main[main.method.eq('ERM-MLP')].set_index('seed')
    sanity,status,gains=oracle_sanity(results);sanity.to_csv(output/'oracle_sanity.csv',index=False)
    fidelity,history=tuning_tables(output)
    chosen=proposed[proposed.proposed_variant_selected].copy()
    chosen[['seed','method','representation','k','dro_variant','validation_selection_score',
            'validation_average_accuracy','accuracy_guard_passed',*METRICS]].to_csv(output/'variant_selection.csv',index=False)
    effects=[]
    for variant,part in proposed.groupby('dro_variant',sort=False):
        ref=baseline.loc[part.seed]
        row={'dro_variant':variant,'n_seeds':len(part),'guard_pass_count':int(part.accuracy_guard_passed.sum())}
        row.update({m:part[m].mean() for m in METRICS})
        for metric in METRICS:
            row[f'delta_{metric}']=part[metric].to_numpy().mean()-ref[metric].to_numpy().mean()
        row['accuracy_within_2pp']=row['delta_accuracy']>=-.02-1e-12
        row['race_gap_reduced']=row['delta_approval_rate_gap_race_eth']<0
        row['sex_gap_reduced']=row['delta_approval_rate_gap_sex']<0
        row['eo_gap_reduced']=row['delta_eo_gap']<0
        row['all_targets_met']=all(row[k] for k in ['accuracy_within_2pp','race_gap_reduced','sex_gap_reduced','eo_gap_reduced'])
        effects.append(row)
    effects=pd.DataFrame(effects);effects.to_csv(output/'proposed_variant_effects.csv',index=False)
    inferred=results[results.representation.isin(['raw','scarf','GEORGE','HBAC','geoproxy'])].copy()
    erm_all=results[results.method.eq('ERM-MLP')].set_index(['selection','seed'])
    for target,metric,sign in [('eo_reduction','eo_gap',-1),('worst_group_gain','worst_group_accuracy',1)]:
        inferred[target]=inferred.apply(lambda r:sign*(r[metric]-erm_all.loc[
            ('average' if r.selection=='inferred_worst_group' else r.selection,r.seed),metric]),axis=1)
    inferred.to_csv(output/'alignment_fairness.csv',index=False)
    correlations=[]
    for (variant,selection),part in inferred.groupby(['dro_variant','selection']):
        for dim in ['race_eth','sex','age','intersection']:
            for metric in ['ari','nmi']:
                key=f'{metric}_{dim}'
                correlations.append({'dro_variant':variant,'selection':selection,'alignment':key,'n':len(part),
                    'pearson_eo_reduction':part[key].corr(part.eo_reduction),
                    'pearson_worst_group_gain':part[key].corr(part.worst_group_gain)})
    corr=pd.DataFrame(correlations);corr.to_csv(output/'alignment_fairness_correlations.csv',index=False)
    text=['# ผลนำร่อง HMDA รอบ 5: นิยามกลุ่มและ class-balanced loss','',
          'ทุกตัวเลขในตารางอ่านจาก CSV; mean ± sample SD ข้าม seed (QUICK 3 seed, full labels); threshold=0.5',
          'EO/worst-group ใช้ intersection race_eth × sex; Unknown และกลุ่มจริงที่เล็กกว่า min_fair_group ไม่นับ fairness; รายละเอียดจำนวนอยู่ใน CSV',
          '## นิยามการทดลองและการเลือกโดยไม่ใช้ประชากร','',
          '- cy/cy_bal: cluster × y; c/c_bal: cluster โดยไม่ผูก y; _bal ใช้ BCE ถ่วงน้ำหนักจาก train เท่านั้น',
          '- raw/SCARF/GEORGE แบบ cy จัด k กลุ่มต่อคลาส (รวม 2k) ตามรอบ 4; แบบ c จัด k กลุ่มบน representation ทั้ง train ไม่แยกคลาส โดยเฉพาะ GEORGE ใช้ ERM hidden representation เดิมแต่ fit global k-means ใหม่',
          '- จึงเปลี่ยนทั้งการใช้ y และจำนวน/องค์ประกอบ cluster ในการเทียบ c กับ cy ไม่ใช่ ablation ที่คง cluster ทุกอย่าง; HBAC/geoproxy/oracle คง base cluster เดิมได้',
          '- oracle: c=race_eth × sex, cy=race_eth × sex × y; oracle เท่านั้นใช้ประชากรเลือก validation; Unknown คงไว้ใน train แต่ไม่นับ validation fairness',
          '- random: c สุ่ม k กลุ่มไม่ใช้ y; cy คงกลุ่มสุ่มต่อคลาสตามรอบ 4; validation ใช้ RNG แยก seed คงที่และไม่ใช้ test',
          '- น้ำหนักภายใน base cluster c: w_i=n_c/(m_c × n_cy), m_c คือจำนวนคลาสที่พบจริง; น้ำหนักแต่ละคลาสรวม n_c/m_c และค่าเฉลี่ยน้ำหนัก=1; หากมีคลาสเดียวได้ w=1',
          '- c_bal คำนวณใน cluster หลังรวมกลุ่มเล็ก; cy_bal คำนวณใน base cluster ก่อน cross y/รวมเล็ก แล้วใช้ mean(w×BCE) ใน DRO group โดยไม่หาร sum(w) ซ้ำ; จึงเป็น cost-sensitive loss และเพิ่มสเกล loss ของ cy ที่เป็นคลาสหายาก ไม่รับรองว่าจะลด approval gap',
          '- cluster ที่ fit ภายในคลาสมี y ค่าเดียวอยู่แล้ว: cy_bal ของ raw/SCARF/GEORGE/random อาจเท่ากับ cy ตามสูตร ไม่ควรตีความเป็นการทดลอง class balance อิสระที่ให้ข้อมูลเพิ่ม; ตรวจ class_weight_min/max ใน CSV',
          '- รวม train group < min_dro_group เข้าศูนย์กลางใกล้สุด; cy รวมเฉพาะคลาสเดียวกัน, c ไม่ใช้ y รวม; validation ใช้ cluster ก่อนรวมเพื่อไม่ซ่อนกลุ่มยาก และนิยาม c/cy ตรงกับ train',
          '- เลือก checkpoint/กริดด้วย inferred_worst_group บน validation ของนิยามนั้น; guard validation accuracy ≥ ERM-MLP −0.02; ถ้าไม่ผ่านทุกตัวเลือก average ที่ใกล้เกณฑ์ที่สุด ตามด้วย worst-group; เสมอคงตัวแรก',
          '- เลือก k และ raw/SCARF ด้วย Silhouette train แยกนิยาม c/cy; จากนั้นเลือกรูปแบบใน 4 ตัวด้วย validation rank เดียวกันโดยไม่ดู test/ประชากร (variant order cy, cy_bal, c, c_bal ใช้ตัดสินเสมอ)',
          '- คะแนน worst-group ของ c กับ cy ต่างจำนวนกลุ่ม/ความยาก จึงเปรียบเทียบข้ามนิยามแล้วมีแนวโน้มเข้าข้าง c; ต้องมี holdout ใหม่ก่อนยืนยัน และอย่าตีความว่ารูปแบบที่เลือกดีที่สุดเชิง fairness จริง',
          '- กริด eta={0.01,0.1} × C={0,1} × lr={0.0002,0.001} × budget=5 epoch (8 candidate); ตัดการฝึก budget=3 ซ้ำ เพราะ checkpoint 1–3 อยู่ใน budget 5 แล้ว ไม่เพิ่ม ERM เป็น candidate; warm-start ERM เฉพาะพารามิเตอร์ optimizer/DRO weights เริ่มใหม่',
          '## ตารางหลัก: วิธี × รูปแบบ','',
          'ERM ใช้ selection=average; JTT/DRO ใช้ inferred_worst_group; guard เป็นสัดส่วน seed ที่ผ่าน (1=ครบ); ไม่มี guard สำหรับ ERM-LGBM แสดง N/A',
          markdown_table(comparison,['method','dro_variant'],METRICS+['accuracy_guard_passed']),'',
          '## วิธีเสนอแยก 4 รูปแบบ และผลเปลี่ยนเทียบ ERM-MLP','',
          markdown_table(effects,['dro_variant'],METRICS+['guard_pass_count','delta_accuracy',
              'delta_approval_rate_gap_race_eth','delta_approval_rate_gap_sex','delta_eo_gap','all_targets_met']),
          'all_targets_met ต้อง accuracy ลดไม่เกิน 0.02 และช่องว่าง race, sex, EO ลดทั้งสามพร้อมกันในค่าเฉลี่ย; ไม่ใช่นัยสำคัญทางสถิติ และ test guard นี้ต่างจาก validation guard',
          '## รูปแบบที่เลือกด้วย validation เท่านั้น (หนึ่งตัวต่อ seed)','',
          markdown_table(chosen,['seed','representation','k','dro_variant'],['validation_selection_score',
              'validation_average_accuracy','accuracy_guard_passed',*METRICS]),
          'หน้าสาธิตใช้ seed แรกและรูปแบบที่เลือกในตารางนี้ การ์ดใช้ cluster definition เดียวกัน; ไม่เลือก seed ด้วย test',
          '### รูปแบบที่เลือกแยกแต่ละวิธี (ไม่รวม oracle)','',
          markdown_table(main[main.variant_selected_without_demographics],['seed','method','dro_variant'],
              ['validation_selection_score','validation_average_accuracy','accuracy_guard_passed']),'',
          '## Sanity oracle แยกรูปแบบ','',
          markdown_table(sanity,['dro_variant'],['erm_worst_group','oracle_worst_group','gain'])]
    for variant in ['cy','cy_bal','c','c_bal']:
        text.append(f"- {variant}: {'ผ่าน' if status[variant] else 'ไม่ผ่าน'}; mean worst-group gain={gains[variant]:.6f}")
    text.extend(['oracle ที่ไม่ผ่านยังไม่ใช่เพดานบนที่ยืนยันแล้ว; การเปลี่ยนพารามิเตอร์จริงไม่รับรอง fairness; กลุ่มฝึก loss, validation worst accuracy และ test intersection เป็นคนละเป้าหมาย',
          '## ผลเสริม: average และ true worst_group (หลังใช้ประชากร validation)','',
          markdown_table(results[results.selection.isin(['average','worst_group'])],['selection','method','dro_variant'],METRICS),'',
          '## กริดที่เลือก: น้ำหนักกลุ่มและการขยับพารามิเตอร์','',
          markdown_table(grid[grid.selected],['selection','seed','method','dro_variant'],
              ['dro_eta','C','learning_rate','selected_epoch','validation_accuracy','validation_selection_score',
               'accuracy_guard_passed','parameter_l2_from_erm','final_group_weight_min','final_group_weight_max']),'',
          '## ความตรงของกลุ่ม (ประชากรบน test ใช้ประเมินหลังเลือกเสร็จ)','',
          markdown_table(clusters,['group_definition','representation','k'],['silhouette','davies_bouldin',
              'ari_race_eth','nmi_race_eth','ari_sex','nmi_sex','ari_age','nmi_age','ari_intersection','nmi_intersection']),
          'Silhouette/DBI ใช้ train ยกเว้น geoproxy รายงานบน test เชิงพรรณนา ไม่ใช้เลือก; HBAC/geoproxy ใช้ cluster ร่วมทั้ง 4 รูปแบบ','',
          '## คำอธิบายกลุ่มและ H2','',
          'ต้นไม้ balanced/unbalanced ลึก ≤3 fit train เลือก F1 validation รายกลุ่ม (เสมอ balanced); fidelity รายงาน validation เดียวที่เลือกจึง optimistic; F1<0.5 เตือนพร้อม SMD/ส่วนต่างสัดส่วน top 5 จาก train',
          'Welch t-test/χ² ใช้ test และ Bonferroni ทั้ง family; HBAC ทำการ์ดแยกนิยามเพื่อหน้าสาธิต แม้ base cluster เหมือนกัน ไม่ถือเป็นหลักฐานอิสระเพิ่ม',
          markdown_table(cards,['group_definition','method'],['fidelity_f1','fidelity_precision','fidelity_recall','rule_length','coverage']),
          markdown_table(fidelity,['round','method'],['n_groups','median_fidelity_f1','fraction_f1_ge_080','n_f1_below_050']),'',
          '## เทียบกับผลรอบก่อน (round4 เก็บสำเนาก่อนรันใหม่)','',
          markdown_table(history[history.method.isin(['ERM-LGBM','ERM-MLP','GDRO-oracle','Proposed (Silhouette)'])],
              ['round','method','dro_variant'],METRICS),'',
          'รอบ 4 oracle มีแต่ intersection บน validation แม้ train cross y; รอบ 5 แก้ validation oracle ให้ตรง c/cy จึงไม่ใช่เปรียบเทียบเกณฑ์เดิมทุกขั้น',
          '## Alignment กับ fairness เชิงสำรวจ','',
          markdown_table(corr,['dro_variant','selection','alignment'],['pearson_eo_reduction','pearson_worst_group_gain']),
          'หลาย k/รูปแบบใช้ข้อมูลและ seed ซ้ำ ไม่เป็นอิสระและไม่แสดงเหตุผลเชิงสาเหตุ','',
          '## ประสิทธิภาพระบบ','',
          markdown_table(results,['selection','method','dro_variant'],['train_seconds','inference_seconds_per_1000',
              'peak_memory_mb','parameters','flops_per_row','tree_count','leaf_count']),
          markdown_table(resources,['representation'],['discovery_train_seconds','peak_memory_mb','parameters','flops_per_row']),
          'เวลาแต่ละแถวรวม preprocessing/discovery ที่ใช้ร่วมกัน ห้ามบวกเป็นเวลาทั้งชุด; FLOPs นับ Linear โมเดลตัดสินใจ; runtime_summary.csv แสดงเวลาทั้งชุด','',
          '## ข้อจำกัด','',
          '- อัตราอนุมัติสูง ต้องอ่าน positive rate/AUROC/F1 ร่วมกับ accuracy; EO ผันผวนเมื่อกลุ่มมีผู้ปฏิเสธน้อย แม้ขนาดรวมผ่าน 30 แถว',
          '- HMDA สาธารณะไม่มีคะแนนเครดิตจริง applicant_credit_score_type เป็นประเภทวิธีคำนวณ; ห้ามอ้างว่าแทนข้อมูลเครดิตครบ',
          '- ใช้ผลสินเชื่อย้อนหลัง ไม่ใช่หลักฐานการเลือกปฏิบัติ; proxy และ Unknown ยังมีข้อจำกัด',
          '- cy class-conditioned clustering ใช้ y ตอน fit แต่ตอนใช้จริง nearest center ไม่รู้ y จึงอาจไม่ตรงกลุ่มฝึก; c ไม่มีปัญหานี้ใน k-means แต่ representation GEORGE และ HBAC ใช้ป้าย train ตามวิธี',
          '- เปิดดู test ซ้ำหลายรอบแล้ว ผลเป็น exploratory ต้องข้อมูล holdout ใหม่; การเลือก variant ด้วย validation ลดการใช้ประชากรแต่ไม่แก้ multiple selection',
          '- ไม่ทดสอบ FULL หรือ Colab รอบ 5 สด; รอบ 2 ผู้สั่งแจ้ง Colab CPU ผ่าน 15 ต.ค. 2569 (~3.4 นาที, Gradio ทำงาน); fallback ใช้แคชในเครื่อง ไม่ทดสอบ download/install/network',
          '- Gradio เติมตัวแปรที่ไม่กรอกจาก median/mode train ความน่าจะเป็นยังไม่ calibrate ใช้เพื่อวิจัยเท่านั้น',''])
    (output/'hmda_pilot_summary.md').write_text('\n'.join(text),encoding='utf-8')
    plot_results(output,results,clusters,cards,proposed)
    if (output/'postprocess_results.csv').exists():
        from .postprocess_reporting import generate_postprocess_report
        generate_postprocess_report(output)


def plot_results(output,results,clusters,cards,proposed):
    """กราฟแยกรูปแบบ ไม่วาดเส้นเชื่อมให้เข้าใจผิดเป็นแนวโน้มเวลา"""
    os.environ.setdefault('MPLCONFIGDIR',str(output/'.matplotlib'))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    figs=output/'figures';figs.mkdir(exist_ok=True)
    main=main_rows(results)
    baseline=main[main.method.eq('ERM-MLP')]
    fig,axes=plt.subplots(2,2,figsize=(14,10),sharex=True,sharey=True)
    for ax,variant in zip(axes.flat,['cy','cy_bal','c','c_bal']):
        for method,g in main[main.dro_variant.eq(variant)].groupby('method'):
            ax.errorbar(g.approval_rate_gap_race_eth.mean(),g.accuracy.mean(),
                        xerr=g.approval_rate_gap_race_eth.std(),yerr=g.accuracy.std(),fmt='o',label=method,alpha=.75)
        p=proposed[proposed.dro_variant.eq(variant)]
        ax.scatter(p.approval_rate_gap_race_eth.mean(),p.accuracy.mean(),marker='*',s=200,color='black',label='Proposed (Silhouette)')
        ax.scatter(baseline.approval_rate_gap_race_eth.mean(),baseline.accuracy.mean(),marker='X',s=100,color='red',label='ERM-MLP')
        ax.set(title=variant,xlabel='Approval gap: race/ethnicity',ylabel='Accuracy');ax.grid(alpha=.2)
    handles,labels=axes[0,0].get_legend_handles_labels()
    fig.legend(handles,labels,loc='center left',bbox_to_anchor=(.8,.5),fontsize=8)
    fig.tight_layout(rect=(0,0,.8,1));fig.savefig(figs/'dro_variants.png',dpi=140);plt.close(fig)
    for selection,part in results.groupby('selection'):
        fig,ax=plt.subplots(figsize=(12,6))
        for (method,variant),g in part.groupby(['method','dro_variant']):
            ax.scatter(g.eo_gap.mean(),g.accuracy.mean(),label=f'{method}/{variant}',s=20)
        ax.set(xlabel='EO gap',ylabel='Accuracy',title=selection)
        ax.legend(fontsize=5,ncol=2,loc='center left',bbox_to_anchor=(1,.5))
        fig.tight_layout();fig.savefig(figs/f'accuracy_eo_{selection}.png',dpi=140);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(12,5))
    for rep,g in clusters.groupby('representation'): axes[0].scatter(g.ari_intersection,g.nmi_intersection,label=rep)
    for (d,method),g in cards.groupby(['group_definition','method']): axes[1].scatter(g.coverage,g.fidelity_f1,label=f'{method}/{d}',s=15)
    axes[0].set(xlabel='ARI intersection',ylabel='NMI intersection');axes[0].legend(fontsize=6)
    axes[1].set(xlabel='Validation rule coverage',ylabel='Validation fidelity F1');axes[1].legend(fontsize=5)
    fig.tight_layout();fig.savefig(figs/'alignment_explanations.png',dpi=140);plt.close(fig)


In [ ]:
%%writefile src/fairgroups/metrics.py
"""ตัวชี้วัดการตัดสินใจและช่องว่างระหว่างกลุ่มจริง"""
import json
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from .data import reference_groups as acs_groups
from .hmda import reference_groups as hmda_groups, UNKNOWN


def group_metrics(y, prediction, group, min_size=1):
    """EO = max(ช่วง TPR, ช่วง FPR); ไม่แทนอัตราที่ไม่มีตัวหารด้วยศูนย์"""
    details, tprs, fprs, positives = {}, [], [], []
    for value in np.unique(group):
        mask = group == value
        if mask.sum() < min_size:
            continue
        yy, pp = y[mask], prediction[mask]
        tpr = float(pp[yy == 1].mean()) if (yy == 1).any() else np.nan
        fpr = float(pp[yy == 0].mean()) if (yy == 0).any() else np.nan
        details[str(value)] = {"n": int(mask.sum()), "accuracy": float((yy == pp).mean()),
                              "tpr": tpr if np.isfinite(tpr) else None,
                              "fpr": fpr if np.isfinite(fpr) else None,
                              "positive_rate": float(pp.mean())}
        if np.isfinite(tpr):
            tprs.append(tpr)
        if np.isfinite(fpr):
            fprs.append(fpr)
        positives.append(float(pp.mean()))
    def gap(values):
        return max(values) - min(values) if len(values) >= 2 else np.nan
    return {"eo_gap": max(gap(tprs), gap(fprs)), "tpr_gap": gap(tprs),
            "fpr_gap": gap(fprs), "dp_difference": gap(positives),
            "worst_group_accuracy": min((v["accuracy"] for v in details.values()), default=np.nan),
            "n_eligible_groups": len(details), "group_accuracy": details}


def evaluate(y, probability, sensitive, min_detail=30, prediction=None):
    """คอลัมน์ fairness หลักใช้ intersection เพศ × White/Non-white"""
    prediction = (probability >= 0.5).astype(int) if prediction is None else np.asarray(prediction, dtype=int)
    result = {"accuracy": accuracy_score(y, prediction),
              "precision": precision_score(y, prediction, zero_division=0),
              "recall": recall_score(y, prediction, zero_division=0),
              "f1": f1_score(y, prediction, zero_division=0),
              "auroc": roc_auc_score(y, probability)}
    refs = hmda_groups(sensitive) if "race_eth" in sensitive else acs_groups(sensitive)
    result["positive_rate"] = float(prediction.mean())
    for name, group in refs.items():
        eligible = group != UNKNOWN
        result[f"excluded_unknown_{name}"] = int((~eligible).sum())
        scores = group_metrics(y[eligible], prediction[eligible], group[eligible], min_detail)
        result[f"excluded_small_{name}"] = int(eligible.sum() - sum(v["n"] for v in scores["group_accuracy"].values()))
        result[f"approval_rate_gap_{name}"] = scores["dp_difference"]
        for metric, value in scores.items():
            result[f"{metric}_{name}"] = json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else value
        if name == "intersection":
            for metric in ["eo_gap", "tpr_gap", "fpr_gap", "dp_difference", "worst_group_accuracy"]:
                result[metric] = scores[metric]
    return result


In [ ]:
%%writefile src/fairgroups/models.py
"""โมเดล MLP และการเลือก checkpoint ด้วย validation เท่านั้น"""
import copy
import random
import numpy as np
import torch
from torch import nn
from .dro import group_dro_loss


def set_seed(seed, threads=2):
    """ใช้ CPU และตั้ง seed เพื่อให้ทำซ้ำได้"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(threads)
    torch.use_deterministic_algorithms(True)


class MLP(nn.Module):
    """ทุกวิธีใช้ input และโครงข่ายตัดสินใจเดียวกันเพื่อเทียบตรง"""
    def __init__(self, input_dim, hidden):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.layers(x).squeeze(-1)


def predict_mlp(model, x):
    """ทำนายเป็นชุดเล็กเพื่อลด RAM"""
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(torch.as_tensor(x[i:i+2048])).sigmoid().numpy()
                               for i in range(0, len(x), 2048)])


def worst_accuracy(correct, groups, min_size=1):
    """ไม่นับ Unknown หรือกลุ่มที่เล็กกว่าเกณฑ์; กลุ่มอนุมานนับทุกกลุ่มที่มีแถว"""
    values = [float(correct[groups == g].mean()) for g in np.unique(groups)
              if str(g) != 'Unknown' and (groups == g).sum() >= min_size]
    if not values:
        raise ValueError('ไม่มี validation group ที่ผ่านเกณฑ์ขนาด')
    return min(values)


def cross_labels(clusters, y):
    """กลุ่ม validation = cluster ที่ตัวจัดกลุ่ม train ทำนาย × ป้าย y"""
    return np.array([f'{c}|y={label}' for c, label in zip(clusters, y)])


def selection_rank(selection, scores, accuracy_floor):
    """ถ้าไม่ผ่าน guard ทุกตัว เลือก average ใกล้ ERM ที่สุดก่อน; เสมอคงตัวแรก"""
    if selection != 'inferred_worst_group':
        return (scores[selection],)
    average = scores['average']; worst = scores[selection]
    passed = average + 1e-12 >= accuracy_floor
    return (passed, worst, average) if passed else (passed, average, worst)


def validation_scores(y, probability, groups):
    """average ไม่แตะกลุ่มจริง ส่วน worst-group เป็นผลเสริมที่ใช้ประชากร"""
    correct = (probability >= 0.5) == y
    return {'average': float(correct.mean()), 'worst_group': worst_accuracy(correct, groups)}


def parameter_distance(model, initial_model):
    """ระยะพารามิเตอร์ L2 จาก ERM เริ่มต้น ไม่รวม buffer"""
    if initial_model is None:
        return 0.0
    return float(sum((a.detach() - b.detach()).double().square().sum()
                     for a, b in zip(model.parameters(), initial_model.parameters())).sqrt())


def train_mlp(x, y, xv, yv, val_groups, config, seed, groups=None,
              sample_weights=None, epochs=None, initial_model=None, learning_rate=None,
              inferred_val_groups=None, accuracy_floor=0.0, inferred_min_size=1):
    """เก็บ checkpoint แต่ละกติกาแยกกัน; guard ใช้กับกติกาหลักของวิธีที่มีกลุ่ม"""
    set_seed(seed, config.threads)
    model = MLP(x.shape[1], config.hidden)
    if initial_model is not None:
        model.load_state_dict(copy.deepcopy(initial_model.state_dict()))
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate or config.learning_rate,
                                 weight_decay=config.weight_decay)
    xt, yt = torch.as_tensor(x), torch.as_tensor(y, dtype=torch.float32)
    gt = None if groups is None else torch.as_tensor(groups, dtype=torch.long)
    wt = None if sample_weights is None else torch.as_tensor(sample_weights, dtype=torch.float32)
    log_weights = None if groups is None else torch.zeros(int(np.max(groups)) + 1)
    group_counts = None if groups is None else torch.bincount(gt, minlength=len(log_weights)).float()
    selections = ["average", "worst_group"]
    if inferred_val_groups is not None:
        selections.append("inferred_worst_group")
    best_ranks = {}
    best_states, best_epochs, best_metadata = {}, {}, {}
    for epoch in range(epochs or config.epochs):
        model.train()
        order = torch.randperm(len(y))
        for ids in order.split(config.batch_size):
            optimizer.zero_grad()
            losses = nn.functional.binary_cross_entropy_with_logits(model(xt[ids]), yt[ids], reduction="none")
            if gt is not None:
                # น้ำหนักเฉลี่ยหนึ่งภายใน cluster ก่อนแยก y; ไม่หาร sum(w) ซ้ำจน cy_bal หายไป
                if wt is not None:
                    losses = losses * wt[ids]
                loss = group_dro_loss(losses, gt[ids], log_weights, config.dro_eta,
                                      group_counts, config.dro_adjustment)
            elif wt is not None:
                # JTT ใช้น้ำหนักคงที่จากโมเดลรอบแรก ไม่เลือกด้วย test
                loss = (losses * wt[ids]).sum() / wt[ids].sum()
            else:
                loss = losses.mean()
            loss.backward()
            optimizer.step()
        probability = predict_mlp(model, xv)
        scores = validation_scores(yv, probability, val_groups)
        if inferred_val_groups is not None:
            scores['inferred_worst_group'] = worst_accuracy(
                (probability >= .5) == yv, inferred_val_groups, inferred_min_size)
        weights = None if log_weights is None else torch.softmax(log_weights, dim=0)
        metadata = {**scores, 'accuracy_floor': accuracy_floor,
                    'accuracy_guard_passed': scores['average'] + 1e-12 >= accuracy_floor,
                    'parameter_l2_from_erm': parameter_distance(model, initial_model),
                    'group_weight_min': float(weights.min()) if weights is not None else np.nan,
                    'group_weight_max': float(weights.max()) if weights is not None else np.nan}
        for selection in selections:
            rank = selection_rank(selection, scores, accuracy_floor)
            if selection not in best_ranks or rank > best_ranks[selection]:
                best_ranks[selection] = rank
                best_states[selection] = copy.deepcopy(model.state_dict())
                best_epochs[selection] = epoch + 1
                best_metadata[selection] = metadata.copy()
    result = {}
    for selection, state in best_states.items():
        selected = MLP(x.shape[1], config.hidden)
        selected.load_state_dict(state)
        selected.selection_metadata_ = best_metadata[selection]
        result[selection] = (selected, best_epochs[selection], best_metadata[selection][selection])
    model.training_diagnostics_ = {
        'final_group_weight_min': float(weights.min()) if weights is not None else np.nan,
        'final_group_weight_max': float(weights.max()) if weights is not None else np.nan,
        'final_parameter_l2_from_erm': parameter_distance(model, initial_model)}
    return result, model


In [ ]:
%%writefile src/fairgroups/postprocess.py
"""ปรับเกณฑ์บน validation เท่านั้น ไม่เปลี่ยนคะแนนเดิม และไม่พึ่ง fairlearn"""
import hashlib
import time
import numpy as np
import pandas as pd
from .efficiency import ResourceMonitor
from .metrics import evaluate


class XRandomGroups:
    """กลุ่มควบคุมจาก hash ของ x: รายเดิมได้กลุ่มเดิม ไม่อ่าน y/ประชากร"""
    def fit(self, x, ids, k, seed):
        self.k, self.seed = k, seed
        self.fit_ids_ = np.asarray(ids).copy()
        return self

    def predict(self, x):
        key = str(self.seed).encode()
        return np.array([int.from_bytes(hashlib.blake2b(row.tobytes(), key=key,
                         digest_size=8).digest(), 'little') % self.k
                         for row in np.asarray(x, dtype='<f4')])


def _span(values):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    return float(np.ptp(values)) if len(values) > 1 else 0.


class ValidationThresholds:
    """Parity ใช้ช่วงอัตราอนุมัติ; EO ใช้ค้นทีละกลุ่ม จึงไม่รับรอง optimum EO"""
    def __init__(self, objective='parity', min_group=50, tolerance=.02):
        if objective not in ('parity', 'eo'):
            raise ValueError('objective ต้องเป็น parity หรือ eo')
        self.objective, self.min_group, self.tolerance = objective, min_group, tolerance

    def _tables(self, p, y, groups):
        tables = []
        for g in self.groups_:
            mask = groups == g
            pg, yg = p[mask], y[mask]
            # ใส่ 0.5 ไว้ก่อนเพื่อคงเกณฑ์เดิมเมื่อผลเสมอกัน
            thresholds = np.array([.5])
            if len(pg) >= self.min_group:
                extra = np.unique(np.r_[np.arange(.05, .951, .01),
                    np.quantile(pg, np.linspace(0, 1, 101)), np.nextafter(pg.max(), np.inf)])
                thresholds = np.r_[.5, extra[extra != .5]]
            pred = pg[:, None] >= thresholds
            tables.append({'threshold': thresholds, 'n': len(pg),
                'correct': (pred == yg[:, None]).sum(axis=0), 'rate': pred.mean(axis=0),
                'tpr': pred[yg == 1].mean(axis=0) if (yg == 1).any() else np.full(len(thresholds), np.nan),
                'fpr': pred[yg == 0].mean(axis=0) if (yg == 0).any() else np.full(len(thresholds), np.nan)})
        return tables

    def _stats(self, indices):
        values = {key: [tab[key][i] for tab, i in zip(self.tables_, indices)]
                  for key in ['correct', 'rate', 'tpr', 'fpr']}
        return {'accuracy': float(sum(values['correct']) / self.n_),
                'approval_gap': _span(values['rate']),
                'eo_gap': max(_span(values['tpr']), _span(values['fpr']))}

    def _parity(self):
        """แต่ละช่วงเลือกจำนวนทายถูกสูงสุดต่อกลุ่ม แล้วตรวจงบ accuracy รวม"""
        lower = np.unique(np.concatenate([t['rate'] for t in self.tables_]))
        def search(width):
            correct, choices = np.zeros(len(lower)), []
            for tab in self.tables_:
                valid = (tab['rate'][None, :] >= lower[:, None] - 1e-12) & (
                         tab['rate'][None, :] <= lower[:, None] + width + 1e-12)
                scores = np.where(valid, tab['correct'][None, :], -self.n_ * 2)
                idx = scores.argmax(axis=1)
                correct += scores[np.arange(len(lower)), idx]
                choices.append(idx)
            feasible = np.flatnonzero(correct >= self.required_correct_)
            if not len(feasible):
                return None
            best = feasible[np.argmax(correct[feasible])]
            return np.array([c[best] for c in choices])
        lo, hi, best = 0., 1., search(1.)
        for _ in range(20):
            mid = (lo + hi) / 2
            candidate = search(mid)
            if candidate is None:
                lo = mid
            else:
                hi, best = mid, candidate
        return best

    def _eo(self, parity):
        """เริ่มจากเกณฑ์เดิมและ parity; รับเฉพาะการขยับที่ผ่าน guard"""
        def rank(indices):
            s = self._stats(indices)
            return (s['eo_gap'], s['approval_gap'], -s['accuracy'])
        candidates = []
        for start in [np.zeros(len(self.groups_), dtype=int), parity]:
            indices = start.copy()
            for _ in range(10):
                before = indices.copy()
                for j, tab in enumerate(self.tables_):
                    correct = sum(t['correct'][i] for t, i in zip(self.tables_, indices)) - tab['correct'][indices[j]]
                    feasible = correct + tab['correct'] >= self.required_correct_
                    keys = []
                    for key in ['tpr', 'fpr', 'rate']:
                        other = np.array([t[key][indices[h]] for h, t in enumerate(self.tables_) if h != j])
                        other = other[np.isfinite(other)]
                        value = tab[key]
                        if len(other):
                            span = np.maximum(np.nan_to_num(value, nan=other.max()), other.max()) - np.minimum(
                                   np.nan_to_num(value, nan=other.min()), other.min())
                        else:
                            span = np.zeros(len(value))
                        keys.append(span)
                    eo = np.maximum(keys[0], keys[1])
                    order = np.lexsort((np.arange(len(eo)), -tab['correct'], keys[2], eo, ~feasible))
                    trial = indices.copy(); trial[j] = order[0]
                    if feasible[order[0]] and rank(trial) < rank(indices):
                        indices = trial
                if np.array_equal(before, indices):
                    break
            candidates.append(indices)
        return min(candidates, key=rank)

    def fit(self, probability, y, groups, ids, *, split='validation'):
        if split != 'validation':
            raise ValueError('หาเกณฑ์ได้เฉพาะ validation')
        p, y, groups = np.asarray(probability), np.asarray(y), np.asarray(groups)
        if not (len(p) == len(y) == len(groups) == len(ids)) or not len(p):
            raise ValueError('ขนาด validation ไม่ตรงกันหรือว่าง')
        if not np.isfinite(p).all() or not np.isin(y, [0, 1]).all():
            raise ValueError('คะแนนต้อง finite และ y เป็น 0/1')
        self.fit_ids_, self.fit_split_ = np.asarray(ids).copy(), split
        self.groups_, self.n_ = np.unique(groups), len(p)
        self.baseline_accuracy_ = float(((p >= .5) == y).mean())
        self.accuracy_floor_ = self.baseline_accuracy_ - self.tolerance
        self.required_correct_ = int(np.ceil(self.accuracy_floor_ * len(y) - 1e-9))
        self.tables_ = self._tables(p, y, groups)
        indices = self._parity()
        if self.objective == 'eo':
            indices = self._eo(indices)
        self.thresholds_ = {g: float(t['threshold'][i]) for g, t, i in zip(self.groups_, self.tables_, indices)}
        self.counts_ = {g: int(t['n']) for g, t in zip(self.groups_, self.tables_)}
        self.validation_ = self._stats(indices)
        assert self.validation_['accuracy'] >= self.accuracy_floor_ - 1e-12
        return self

    def predict(self, probability, groups):
        thresholds = np.array([self.thresholds_.get(g, .5) for g in groups])
        return (np.asarray(probability) >= thresholds).astype(int)


def selection_key(row):
    """อ่านเฉพาะผล validation ของกลุ่มค้นพบ; ไม่รับผล test/กลุ่มจริง"""
    return (not row['accuracy_guard_passed'], row['validation_approval_gap'],
            -row['validation_accuracy'], row['validation_eo_gap'], row['method'], row['objective'])


def fit_candidates(base, probabilities, val, methods, config, seed):
    """เลือกทุกตัวให้เสร็จก่อนส่ง test เข้ามาประเมิน"""
    candidates = []
    for method, rep, k, labels in methods:
        for objective in ['parity', 'eo']:
            with ResourceMonitor() as monitor:
                model = ValidationThresholds(objective, config.min_dro_group,
                    config.selection_accuracy_tolerance).fit(probabilities, val['y'], labels['val'], val['ids'])
            s = model.validation_
            row = {'seed': seed, 'base': base, 'method': method, 'representation': rep, 'k': k,
                'objective': objective, 'fit_split': 'validation', 'group_definition': 'c',
                'uses_demographics_at_inference': rep == 'oracle',
                'validation_accuracy': s['accuracy'], 'validation_approval_gap': s['approval_gap'],
                'validation_eo_gap': s['eo_gap'], 'validation_accuracy_floor': model.accuracy_floor_,
                'erm_validation_accuracy': model.baseline_accuracy_,
                'accuracy_guard_passed': s['accuracy'] >= model.accuracy_floor_ - 1e-12,
                'threshold_fit_seconds': monitor.seconds, 'threshold_peak_process_rss_mb': monitor.peak_mb,
                'threshold_peak_rss_increase_mb': (monitor.peak - monitor.peak_start) / 1024**2,
                'n_validation_groups': len(model.groups_),
                'n_frozen_small_groups': sum(n < config.min_dro_group for n in model.counts_.values()),
                'selected_without_demographics': False, 'proposed_selected': False}
            candidates.append((row, model, labels))
    for flag, eligible in [
        ('selected_without_demographics', [c for c in candidates if c[0]['representation'] not in ['oracle', 'random']]),
        ('proposed_selected', [c for c in candidates if c[0]['representation'] in ['raw', 'scarf']])]:
        min(eligible, key=lambda c: selection_key(c[0]))[0][flag] = True
    return candidates


def evaluate_candidates(candidates, probability, test, min_detail):
    """ขั้นนี้อ่าน test เพื่อวัดผลเท่านั้น ไม่มีการ fit หรือเลือกเพิ่ม"""
    rows, thresholds = [], []
    for metadata, model, labels in candidates:
        start = time.perf_counter()
        prediction = model.predict(probability, labels['test'])
        seconds = time.perf_counter() - start
        rows.append({**metadata, **evaluate(test['y'], probability, test['s'], min_detail, prediction),
            'threshold_inference_seconds_per_1000': seconds * 1000 / len(prediction)})
        for g, threshold in model.thresholds_.items():
            thresholds.append({**{k: metadata[k] for k in ['seed', 'base', 'method', 'objective',
                'proposed_selected', 'selected_without_demographics']}, 'group': g, 'threshold': threshold,
                'n_validation': model.counts_[g], 'frozen_small_group': model.counts_[g] < model.min_group,
                'fit_split': model.fit_split_, 'unseen_group_threshold': .5})
    return rows, thresholds


In [ ]:
%%writefile src/fairgroups/postprocess_reporting.py
"""สรุปรอบ 6 จาก CSV และคงรายงาน DRO เดิมไว้"""
import pandas as pd
from .reporting import markdown_table

METRICS = ['accuracy', 'f1', 'auroc', 'positive_rate', 'approval_rate_gap_race_eth',
           'approval_rate_gap_sex', 'eo_gap', 'worst_group_accuracy']


def sanity_table(frame):
    baseline=frame[frame.objective.eq('baseline')].set_index(['base','seed'])
    rows=[]
    for r in frame[(frame.objective.eq('parity') & frame.representation.eq('oracle')) |
                   frame.representation.eq('random') | frame.proposed_selected].itertuples():
        ref=baseline.loc[(r.base,r.seed)]
        rows.append({'base':r.base,'seed':r.seed,'method':r.method,'objective':r.objective,
            'representation':r.representation,'race_gap_reduction':ref.approval_rate_gap_race_eth-r.approval_rate_gap_race_eth,
            'accuracy_delta':r.accuracy-ref.accuracy,
            'race_gap_reduced':r.approval_rate_gap_race_eth < ref.approval_rate_gap_race_eth})
    return pd.DataFrame(rows)


def generate_postprocess_report(output):
    frame=pd.read_csv(output/'postprocess_results.csv')
    selected=frame[frame.proposed_selected | frame.selected_without_demographics].copy()
    selected.to_csv(output/'postprocess_selection.csv',index=False)
    sanity=sanity_table(frame);sanity.to_csv(output/'postprocess_sanity.csv',index=False)
    proposed=frame[frame.proposed_selected].copy();proposed['method']='Proposed (validation)'
    deployed=frame[frame.selected_without_demographics].copy();deployed['method']='Selected non-oracle (validation)'
    comparison=pd.concat([frame,proposed,deployed],ignore_index=True)
    # ตัวที่เลือกอาจต่าง objective ข้าม seed จึงรวมเป็นแถวเดียวต่อฐาน
    comparison.loc[comparison.method.isin(['Proposed (validation)','Selected non-oracle (validation)']),'objective']='selected'
    baseline=frame[frame.objective.eq('baseline')].set_index(['base','seed'])
    effects=[]
    for r in frame[frame.proposed_selected].itertuples():
        ref=baseline.loc[(r.base,r.seed)]
        effects.append({'base':r.base,'seed':r.seed,'method':r.method,'objective':r.objective,
            **{f'delta_{m}':getattr(r,m)-ref[m] for m in METRICS}})
    effects=pd.DataFrame(effects);effects.to_csv(output/'postprocess_selected_effects.csv',index=False)
    pp=frame[~frame.objective.eq('baseline')]
    text=['## ปรับเกณฑ์รายกลุ่ม','',
        'รอบ 6 เพิ่ม PP บนคะแนน ERM-LGBM/ERM-MLP ที่เลือก checkpoint ด้วย validation average accuracy เดิม; ไม่ฝึกฐานใหม่เพื่อ PP',
        'ทุกตารางอ่านจาก CSV; mean ± sample SD ข้าม seed; gap ใช้หน่วยสัดส่วน (0.01 = 1 จุดร้อยละ)',
        '- raw/SCARF ใช้ global cluster c ที่ fit train เท่านั้น k=2,4,8; GEORGE ใช้ hidden ERM และเลือก k ด้วย silhouette train; HBAC ใช้ OOF error train; geoproxy ใช้สัดส่วนประชากรระดับพื้นที่ (เป็น baseline ที่อาศัย proxy เชื้อชาติ)',
        '- random-k ใช้ hash แบบมี seed ของ x ที่แปลงแล้ว mod k ไม่ใช้ y/ประชากร และให้กลุ่มเดิมได้ตอนอนุมาน; รวม k=2,4,8 เทียบขนาดกลุ่ม raw/SCARF; ไม่รับรองว่ากลุ่มสุ่มจะไร้ความสัมพันธ์โดยบังเอิญในชุดเล็ก',
        '- PP-oracle ใช้ race_eth × sex ตอนหาเกณฑ์และตอนทำนาย เป็นเพดานบนเชิงกลไกเท่านั้น ไม่ใช่วิธีใช้จริง; Unknown เป็นกลุ่มแยกในการ fit แต่ไม่นับ fairness ตามกฎเดิม',
        '- เกณฑ์ fit เฉพาะ validation: กริด 0.05–0.95 ทีละ 0.01 รวม quantile 101 จุด, 0.5 และเกณฑ์ปฏิเสธทั้งหมด; กลุ่ม validation < min_dro_group=50 ถูกตรึง 0.5; กลุ่มใหม่ไม่พบใช้ 0.5',
        '- Parity ค้นช่วงอัตราอนุมัติด้วย bisection 20 ครั้ง เลือกจำนวนทายถูกสูงสุดที่ทำได้ในแต่ละช่วง; หาช่องว่างต่ำสุดบนกริดภายในความละเอียดประมาณ 1e-6',
        '- EO ลด max(TPR gap,FPR gap) ด้วย coordinate descent ไม่เกิน 10 sweeps เริ่มทั้ง 0.5 และ parity; ไม่รับรอง global optimum; อัตราที่ไม่มีตัวหารไม่นับ',
        '- ทั้งสองเป้าหมายบังคับ validation accuracy ≥ ฐานเดียวกันที่ threshold 0.5 − 0.02; มีเกณฑ์เดิมเป็น feasible fallback เสมอ จึงไม่ต้องเลือกตัวที่ตก guard',
        '- เลือกแยกแต่ละฐาน/seed จาก validation approval gap ระหว่างกลุ่มค้นพบ ตามด้วย accuracy สูงกว่า, EO ต่ำกว่า, ชื่อวิธี/เป้าหมายเมื่อเสมอ; ไม่ใช้ test หรือประชากรจริงเลือก',
        '- รายงานตัวเลือกสองชุด: proposed จำกัด raw/SCARF และตัวเลือก non-oracle รวม raw/SCARF/GEORGE/HBAC/geoproxy; random และ oracle ถูกกันออกจากการเลือกทั้งสองชุด',
        '- test ใช้ประชากรจริงเพื่อประเมินเท่านั้น; accuracy/F1/positive rate/EO ใช้คำตัดสินหลังปรับเกณฑ์ แต่ AUROC ใช้คะแนนเดิม จึงเท่าฐานทุกวิธีภายใน seed',
        '### ERM เทียบ PP ทุกวิธีและทุกเป้าหมาย (เฉลี่ย 3 seed ใน QUICK)','',
        markdown_table(comparison,['base','method','objective'],METRICS),'',
        '### ตัวที่เลือกด้วย validation แยกตาม seed','',
        markdown_table(selected,['base','seed','method','objective'],['proposed_selected',
            'selected_without_demographics','validation_approval_gap','validation_accuracy',
            'validation_accuracy_floor','accuracy_guard_passed',*METRICS]),'',
        '### ผลของวิธีเสนอที่เลือกแล้ว เทียบฐานเดียวกัน','',
        markdown_table(effects,['base'],[f'delta_{m}' for m in METRICS]),'',
        '### Sanity checks','',
        markdown_table(sanity,['base','method','objective'],['race_gap_reduction','accuracy_delta','race_gap_reduced'])]
    for base,part in sanity[sanity.representation.eq('oracle')].groupby('base'):
        passed=part.race_gap_reduction.mean()>0
        text.append(f"- oracle parity {base}: {'ผ่าน' if passed else 'ไม่ผ่าน'}; mean race gap reduction={part.race_gap_reduction.mean():.6f}; seed ลดได้ {int(part.race_gap_reduced.sum())}/{len(part)}")
    for base,part in effects.groupby('base'):
        failed=part[part.delta_accuracy < -.02-1e-12]
        text.append(f"- proposed {base}: mean test accuracy delta={part.delta_accuracy.mean():.6f}; test ลดเกิน 2 จุดร้อยละ {len(failed)}/{len(part)} seed (seed={failed.seed.tolist()}); validation guard ของตัวที่เลือกผ่านครบ")
    random=sanity[sanity.representation.eq('random')]
    for (base,objective),part in random.groupby(['base','objective']):
        text.append(f'- random {objective} {base}: mean race gap reduction={part.race_gap_reduction.mean():.6f}; ลดได้ {int(part.race_gap_reduced.sum())}/{len(part)} คู่ seed/k; ข้อมูลซ้ำข้าม k ไม่ใช่ตัวอย่างอิสระ จึงยังยืนยันการลดอย่างเป็นระบบไม่ได้')
    text.extend(['วิธีเสนอเทียบ random/oracle อ่านจากตารางเดียวกัน แยกฐานและเป้าหมาย; ห้ามเลือก k ของ random ด้วย test เพื่อทำคู่เทียบ',
        '### เวลาและหน่วยความจำเฉพาะขั้นปรับเกณฑ์','',
        markdown_table(pp,['base','method','objective'],['threshold_fit_seconds',
            'threshold_inference_seconds_per_1000','threshold_peak_process_rss_mb','threshold_peak_rss_increase_mb',
            'n_frozen_small_groups']),
        f"เวลารวม fit ทุกเกณฑ์ = {pp.threshold_fit_seconds.sum():.3f} วินาที; peak RSS process = {pp.threshold_peak_process_rss_mb.max():.2f} MiB; RSS เพิ่มสูงสุดระหว่าง fit = {pp.threshold_peak_rss_increase_mb.max():.2f} MiB",
        'RSS สุ่มทุก 10 ms รวม process ของฐานเดิม ไม่ใช่ขนาดโมเดล PP; เวลาอนุมาน PP นับเฉพาะ lookup/เปรียบเทียบเกณฑ์ ไม่รวมคำนวณคะแนนและค้นกลุ่มซึ่งใช้ pipeline เดิม',
        '### ข้อจำกัดรอบ 6','',
        '- guard เป็นเงื่อนไขบน validation ไม่รับรองว่า test accuracy จะลดไม่เกิน 2 จุด; การเลือกและ fit เกณฑ์บน validation เดียวกันอาจ optimistic',
        '- การทำ parity ของกลุ่มที่ค้นพบไม่รับรอง parity ของเชื้อชาติ/เพศจริง; ความตรงของกลุ่มเดิมต่ำ และ EO อาจแย่ลงแม้ approval gap ลด',
        '- random มีเพียง 3 seed และกลุ่มเล็กถูกตรึง 0.5 จึงไม่มีข้อรับรองเชิงสถิติ; oracle อาจยังมีช่องว่าง test จาก sampling/guard/กลุ่มเล็ก',
        '- test ชุดเดิมถูกเปิดดูหลายรอบ ผลทั้งหมดเป็น exploratory ต้องยืนยันด้วย holdout ใหม่; ไม่รัน FULL หรือทดสอบ Colab รอบ 6 สด; notebook fallback รันในเครื่องด้วยแคช ไม่ทดสอบติดตั้ง/เครือข่าย',
        '- หน้าสาธิตคงผล DRO เดิมและเพิ่มผล PP จากฐาน LGBM ของ proposed ที่เลือกด้วย validation seed แรก; คะแนนเดิมไม่เปลี่ยน มีเพียงเกณฑ์และคำตัดสินเปลี่ยน',''])
    summary=output/'hmda_pilot_summary.md'
    previous=summary.read_text().split('## ปรับเกณฑ์รายกลุ่ม')[0].rstrip()
    summary.write_text(previous+'\n\n'+'\n'.join(text),encoding='utf-8')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig,axes=plt.subplots(1,2,figsize=(15,6),sharex=True,sharey=True)
    for ax,base in zip(axes,['ERM-LGBM','ERM-MLP']):
        for (method,objective),part in comparison[comparison.base.eq(base)].groupby(['method','objective']):
            mark='*' if objective=='selected' else 'X' if objective=='baseline' else 'o' if objective=='parity' else '^'
            ax.scatter(part.approval_rate_gap_race_eth.mean(),part.accuracy.mean(),marker=mark,
                s=130 if mark in ['*','X'] else 35,label=f'{method}/{objective}',alpha=.8)
        ax.set(title=base,xlabel='Approval-rate gap: race/ethnicity',ylabel='Accuracy');ax.grid(alpha=.2)
    axes[-1].legend(fontsize=6,loc='center left',bbox_to_anchor=(1,.5))
    fig.tight_layout();fig.savefig(output/'figures/postprocess_tradeoff.png',dpi=140);plt.close(fig)


In [ ]:
%%writefile src/fairgroups/reporting.py
"""สร้างรายงานและกราฟจาก CSV ที่บันทึกแล้วเท่านั้น"""
import os
from pathlib import Path
import numpy as np
import pandas as pd


def markdown_table(frame, groups, metrics):
    """ไม่ใช้ tabulate เพื่อลดแพ็กเกจที่ต้องติดตั้ง"""
    rows = ["| " + " | ".join(groups + metrics) + " |",
            "| " + " | ".join(["---"] * (len(groups) + len(metrics))) + " |"]
    for key, part in frame.groupby(groups, sort=True, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        cells = [str(v) for v in key]
        for metric in metrics:
            values = part[metric].dropna()
            if len(values) > 1:
                cells.append(f"{values.mean():.4f} ± {values.std(ddof=1):.4f}")
            else:
                cells.append(f"{values.mean():.4f}" if len(values) else "N/A")
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join(rows)


def generate_reports(root):
    """โหลดผลจากดิสก์ใหม่ ไม่ฝังตัวเลขผลทดลองลงในข้อความ"""
    root = Path(root)
    results = pd.read_csv(root / "pilot_results.csv")
    clusters = pd.read_csv(root / "clustering_results.csv")
    resources = pd.read_csv(root / "representation_resources.csv")
    groups = ["selection", "label_fraction", "method"]
    text = ["# ผลนำร่อง ACS CA 2018", "",
            "ค่าทุกตารางเป็น mean ± sample SD ข้าม seed และอ่านจาก CSV เท่านั้น", "",
            "selection=average เป็นผลหลัก; worst_group ใช้เพศ/เชื้อชาติจริงใน validation เพื่อเลือก checkpoint", "",
            "EO gap และ worst-group หลักใช้เพศ × White/Non-white; เกณฑ์ทำนาย 0.5 ทุกวิธี", "",
            "## ความแม่นยำและความเป็นธรรม", "",
            markdown_table(results, groups, ["accuracy", "precision", "recall", "f1", "auroc",
                           "eo_gap", "dp_difference", "worst_group_accuracy"]), "",
            "## ความตรงของกลุ่ม (วัดบนข้อมูล train ที่ใช้ค้นหากลุ่ม)", "",
            "raw/scarf ใช้ train ทั้งหมดโดยไม่ใช้ y; error ใช้ OOF เฉพาะ train ที่มีป้ายกำกับตามงบ", "",
            "discovery_label_fraction=0 หมายถึงไม่ใช้ป้ายกำกับ ไม่ใช่ไม่มีข้อมูลฝึก", "",
            markdown_table(clusters, ["representation", "discovery_label_fraction", "k"],
                           ["silhouette", "davies_bouldin", "ari_sex", "nmi_sex", "ari_race", "nmi_race",
                            "ari_intersection", "nmi_intersection"]), "",
            "Reconstruction error: N/A เพราะ SCARF ไม่มี decoder", "",
            "## ประสิทธิภาพระบบ", "",
            "train_seconds รวม preprocessing + discovery ที่นำกลับใช้ร่วมกัน + การฝึกตัวตัดสินใจของวิธีนั้น "
            "จึงห้ามนำเวลาแต่ละแถวมารวมเป็นเวลารันทั้งชุด; peak_memory_mb เป็น RSS รวมของ Python ที่สุ่มทุก 10 ms", "",
            markdown_table(results, groups, ["train_seconds", "decision_train_seconds",
                           "inference_seconds_per_1000", "peak_memory_mb", "parameters", "flops_per_row",
                           "tree_count", "leaf_count"]), "",
            "## ต้นทุนการค้นหา representation", "",
            "SCARF encoder แยกจาก projection head; FLOPs นับเฉพาะ Linear ตาม 2×in×out", "",
            markdown_table(resources, ["representation", "discovery_label_fraction"],
                           ["discovery_train_seconds", "inference_seconds_per_1000", "peak_memory_mb", "parameters", "flops_per_row",
                            "pretraining_parameters", "pretraining_flops_per_row"]), "",
            "## ความสัมพันธ์ของความตรงของกลุ่มกับผลความเป็นธรรม", "",
            "ค่าสหสัมพันธ์เชิงสำรวจจาก cluster methods เท่านั้น ยังไม่ใช่ข้อสรุปเชิงเหตุและผล", ""]
    relation_rows = []
    for (selection, fraction), part in results[results.representation.isin(["raw", "scarf", "error"]) &
                                                    results.method.str.startswith("GDRO-")].groupby(["selection", "label_fraction"]):
        for metric in ["ari_intersection", "nmi_intersection"]:
            relation_rows.append({"selection": selection, "label_fraction": fraction, "alignment": metric,
                "pearson_worst_group": part[metric].corr(part.worst_group_accuracy),
                "pearson_eo_gap": part[metric].corr(part.eo_gap), "n": len(part)})
    relation = pd.DataFrame(relation_rows)
    relation.to_csv(root / "alignment_fairness_correlations.csv", index=False)
    text += [markdown_table(relation, ["selection", "label_fraction", "alignment"],
                            ["pearson_worst_group", "pearson_eo_gap"]), "",
             "## ข้อจำกัด", "",
             "- เป็น pilot ขนาดย่อและ epochs น้อย ผลไม่ได้รับประกันว่าจะลดช่องว่างได้ทุกวิธี", 
             "- error เป็น label-assisted discovery จึงแยกจาก raw/scarf ที่ไม่ใช้ป้ายกำกับ",
             "- กลุ่มละเอียดขนาดต่ำกว่าเกณฑ์ถูกตัดออกจาก fairness แบบละเอียดเท่านั้น ดูจำนวนใน CSV",
             "- ใช้ POBP ซึ่งอาจเป็น proxy เชื้อชาติ; oracle ใช้ข้อมูลประชากร train และเป็นเพียงตัวเทียบ",
             "- การแบ่งข้อมูล stratified ใช้กลุ่มจริงเพื่อออกแบบการประเมิน; ตัวฝึกหลักไม่รับข้อมูลนั้น",
             "- ยังไม่ยืนยันเวลาจริงบน Colab; ARL และ Gradio อยู่ใน TODO", ""]
    # EO ต่ำอาจเกิดจากทายคลาสเดียว จึงตรวจจากอัตราทายบวกที่เก็บจริงใน CSV
    for (selection, fraction), part in results[results.method == "JTT"].groupby(["selection", "label_fraction"]):
        rates = []
        for raw in part.group_accuracy_intersection:
            import json
            groups = json.loads(raw)
            rates.append(sum(g["n"] * g["positive_rate"] for g in groups.values()) /
                         sum(g["n"] for g in groups.values()))
        if any(rate > 0.98 or rate < 0.02 for rate in rates):
            text.append(f"- ตรวจพบ JTT ทำนายกระจุกคลาสเดียวอย่างน้อยหนึ่ง seed: selection={selection}, "
                        f"labels={fraction:g}, อัตราทายบวกเฉลี่ย={np.mean(rates):.4f}; "
                        "EO gap ต่ำในกรณีนี้ไม่ใช่หลักฐานว่าตัดสินใจได้ดีขึ้น ต้องอ่าน accuracy/recall ควบคู่")
    text.append("")
    (root / "pilot_summary.md").write_text("\n".join(text), encoding="utf-8")
    make_figures(results, clusters, root)


def make_figures(results, clusters, root):
    """กราฟแยกกติกาเลือกโมเดลเพื่อไม่ปะปนการใช้ข้อมูลประชากร"""
    os.environ.setdefault("MPLCONFIGDIR", str(root / ".matplotlib"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    figure_dir = root / "figures"
    figure_dir.mkdir(exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for metric, ax in zip(["ari_intersection", "nmi_intersection"], axes):
        for (rep, frac), part in clusters.groupby(["representation", "discovery_label_fraction"]):
            values = part.groupby("k")[metric].agg(["mean", "std"])
            ax.errorbar(values.index, values["mean"], yerr=values["std"], marker="o", label=f"{rep} labels={frac:g}")
        ax.set(xlabel="k", ylabel=metric, xticks=[2, 4, 8])
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(figure_dir / "alignment.png", dpi=160)
    plt.close(fig)
    for selection, selected in results.groupby("selection"):
        fractions = sorted(selected.label_fraction.unique())
        # ให้ทั้ง 16 วิธีมีสีต่างกัน และใช้สัญลักษณ์ช่วยอ่านกรณีพิมพ์ขาวดำ
        method_names = sorted(selected.method.unique())
        colors = {name: plt.get_cmap("tab20")(i) for i, name in enumerate(method_names)}
        markers = {name: ["o", "s", "^", "D", "v", "P", "X", "*"][i % 8]
                   for i, name in enumerate(method_names)}
        fig, axes = plt.subplots(1, len(fractions), figsize=(7*len(fractions), 5), squeeze=False)
        for fraction, ax in zip(fractions, axes[0]):
            part = selected[selected.label_fraction == fraction]
            for method, group in part.groupby("method"):
                ax.errorbar(group.eo_gap.mean(), group.accuracy.mean(),
                            xerr=group.eo_gap.std(), yerr=group.accuracy.std(),
                            fmt=markers[method], color=colors[method], label=method)
            ax.set(xlabel="EO gap (lower is better)", ylabel="Accuracy", title=f"Labels {fraction:.0%} / {selection}")
        axes[0, -1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
        fig.tight_layout()
        fig.savefig(figure_dir / f"accuracy_eo_{selection}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(11, 5))
        for method, part in selected.groupby("method"):
            values = part.groupby("label_fraction").worst_group_accuracy.agg(["mean", "std"])
            ax.errorbar(values.index, values["mean"], yerr=values["std"],
                        marker=markers[method], color=colors[method], label=method)
        ax.set(xlabel="Label fraction", ylabel="Worst-group accuracy", title=selection, xticks=fractions)
        ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
        fig.tight_layout()
        fig.savefig(figure_dir / f"worst_group_labels_{selection}.png", dpi=160)
        plt.close(fig)


In [ ]:
%%writefile src/fairgroups/representations.py
"""SCARF ไม่มีป้ายกำกับ และ error representation จาก OOF ภายใน train"""
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from .models import set_seed


class ScarfEncoder(nn.Module):
    """สุ่มแทนค่าทีละ feature จาก marginal ของ train ก่อนทำ one-hot"""
    def __init__(self, input_dim, hidden, embedding):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU(),
                                     nn.Linear(hidden, embedding))
        self.projector = nn.Sequential(nn.Linear(embedding, embedding), nn.ReLU(),
                                       nn.Linear(embedding, embedding))

    def forward(self, x):
        return self.encoder(x)


def contrastive_loss(a, b, temperature):
    """NT-Xent: คู่จริงคือแถวเดิมกับแถวที่แทนค่าบาง feature"""
    n = len(a)
    z = nn.functional.normalize(torch.cat([a, b]), dim=1)
    similarity = z @ z.T / temperature
    similarity.fill_diagonal_(-1e9)
    targets = (torch.arange(2*n) + n) % (2*n)
    return nn.functional.cross_entropy(similarity, targets)


def train_scarf(frame, processor, config, seed):
    """ไม่รับ y หรือข้อมูลประชากร และ donor มาจาก train เท่านั้น"""
    set_seed(seed, config.threads)
    x = processor.transform(frame)
    model = ScarfEncoder(x.shape[1], config.hidden, config.embedding)
    model.fit_ids_ = frame.index.to_numpy().copy()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                                 weight_decay=config.weight_decay)
    rng = np.random.default_rng(seed)
    # สร้าง corrupted view ครั้งละ epoch เพื่อให้หมวดหมู่ยังเป็นค่าที่ถูกต้อง
    for _ in range(config.scarf_epochs):
        corrupted = frame.copy()
        for col in frame.columns:
            donors = rng.integers(0, len(frame), len(frame))
            mask = rng.random(len(frame)) < config.corruption_rate
            corrupted.loc[mask, col] = frame[col].to_numpy()[donors[mask]]
        cx = processor.transform(corrupted)
        order = rng.permutation(len(x))
        model.train()
        for start in range(0, len(x), config.batch_size):
            ids = order[start:start+config.batch_size]
            if len(ids) < 2:
                continue
            optimizer.zero_grad()
            a = model.projector(model(torch.as_tensor(x[ids])))
            b = model.projector(model(torch.as_tensor(cx[ids])))
            loss = contrastive_loss(a, b, config.temperature)
            loss.backward()
            optimizer.step()
    return model


def encode_scarf(model, x):
    """แช่แข็ง encoder ตอนจัดกลุ่ม ไม่ใช้ projection head"""
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(torch.as_tensor(x[i:i+2048])).numpy()
                               for i in range(0, len(x), 2048)])


def crossfit_error(x, y, ids, config, seed):
    """ใช้ป้ายกำกับเท่าที่มี และตรวจว่าแถวที่ทายไม่เคยเข้าโมเดล fold นั้น"""
    probability = np.full(len(y), np.nan)
    coverage = np.zeros(len(y), dtype=int)
    folds = []
    splitter = StratifiedKFold(n_splits=config.oof_folds, shuffle=True, random_state=seed)
    for train, held in splitter.split(x, y):
        assert set(ids[train]).isdisjoint(ids[held])
        model = LGBMClassifier(n_estimators=config.trees, num_leaves=15, max_depth=5,
                               verbosity=-1, n_jobs=config.threads, random_state=seed,
                               deterministic=True, force_col_wise=True)
        model.fit(x[train], y[train])
        probability[held] = model.predict_proba(x[held])[:, 1]
        coverage[held] += 1
        folds.append({"fit_ids": ids[train].tolist(), "heldout_ids": ids[held].tolist()})
    assert np.all(coverage == 1) and np.isfinite(probability).all()
    p = np.clip(probability, 1e-6, 1-1e-6)
    loss = -(y * np.log(p) + (1-y) * np.log(1-p))
    scaler = StandardScaler()
    representation = scaler.fit_transform(np.column_stack([loss, y, p])).astype(np.float32)
    return representation, {"fit_ids": ids.tolist(), "folds": folds,
                            "oof_coverage_min": int(coverage.min()),
                            "oof_coverage_max": int(coverage.max())}


In [ ]:
%%writefile src/fairgroups/run.py
"""HMDA pilot ต่อยอดโมดูลเดิม: train-only discovery และ full labels เท่านั้น"""
import json
import time
import warnings
from dataclasses import replace
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from .hmda import load_hmda, split_hmda, LeakageGuard, HMDAPreprocessor, write_profile
from .hmda_groups import ClassClusters, GlobalClusters, GeoGroups, hidden, fit_hbac, group_scores
from .models import set_seed, train_mlp, predict_mlp, validation_scores, cross_labels, selection_rank
from .representations import train_scarf, encode_scarf
from .efficiency import ResourceMonitor, network_cost, inference_seconds, environment
from .metrics import evaluate
from .tracking import setup_tracking, log_row
from .explanations import describe_groups, export_explanations
from .dro import make_dro_groups, balanced_cluster_weights, validation_groups
from .postprocess import XRandomGroups, fit_candidates, evaluate_candidates


def save_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8')


def oof_errors(train,config,seed):
    """แต่ละ fold มี imputer/encoder ของตัวเอง แถว held ไม่เข้า fit แม้ขั้นแปลง"""
    y=train['y']; p=np.full(len(y),np.nan); folds=[]
    for fit,held in StratifiedKFold(config.oof_folds,shuffle=True,random_state=seed).split(train['x'],y):
        # ตัวกรอง missing gap อาศัย y จึงต้อง fit ใหม่ภายใน fold ด้วย
        fold_guard=LeakageGuard().fit(train['frame'].iloc[fit],y[fit])
        fold_train=fold_guard.transform(train['frame'].iloc[fit])
        fold_held=fold_guard.transform(train['frame'].iloc[held])
        pre=HMDAPreprocessor(config.min_category_frequency,config.lei_top_n).fit(fold_train)
        model=LGBMClassifier(n_estimators=config.trees,num_leaves=15,max_depth=5,verbosity=-1,
                             n_jobs=config.threads,random_state=seed,deterministic=True,force_col_wise=True)
        model.fit(pre.transform(fold_train),y[fit])
        p[held]=model.predict_proba(pre.transform(fold_held))[:,1]
        folds.append({'fit_ids':train['ids'][fit].tolist(),'heldout_ids':train['ids'][held].tolist(),
                      'guard_fit_ids':fold_guard.fit_ids_.tolist(),'preprocessor_fit_ids':pre.fit_ids_.tolist(),
                      'features':fold_guard.features_})
    assert np.isfinite(p).all()
    return ((p>=.5)!=y).astype(float),{'fit_ids':train['ids'].tolist(),'folds':folds}


def choose_jtt(xt,train,xv,val,vg,config,seed,erm_accuracy=0.0):
    """JTT มีกลุ่มผิด/ถูกจากโมเดลรอบสั้น × y; เลือกแยกทั้งสามเกณฑ์"""
    _, identification=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed,epochs=config.jtt_identification_epochs)
    mistakes=(predict_mlp(identification,xt)>=.5)!=train['y']
    val_mistakes=(predict_mlp(identification,xv)>=.5)!=val['y']
    inferred=cross_labels(val_mistakes.astype(int),val['y'])
    floor=erm_accuracy-config.selection_accuracy_tolerance
    chosen={}; best_ranks={}; history=[]
    for weight in config.jtt_grid:
        checkpoints,_=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed,
            sample_weights=np.where(mistakes,weight,1.),inferred_val_groups=inferred,accuracy_floor=floor)
        for selection,cp in checkpoints.items():
            model,epoch,score=cp; meta=model.selection_metadata_
            rate=float((predict_mlp(model,xv)>=.5).mean())
            row={'seed':seed,'selection':selection,'upweight':weight,
                 'validation_accuracy':meta['average'],'validation_selection_score':score,
                 'selected_epoch':epoch,'positive_rate':rate,'collapsed':rate<=.005 or rate>=.995,
                 'n_mistakes':int(mistakes.sum()),'accuracy_guard_passed':meta['accuracy_guard_passed'],
                 'accuracy_floor':floor,'selected':False}
            history.append(row)
            rank=selection_rank(selection,meta,floor)
            if selection not in best_ranks or rank>best_ranks[selection]:
                chosen[selection]=cp;best_ranks[selection]=rank
                model.selection_metadata_={**meta,'jtt_upweight':weight}
                chosen[selection][0].grid_index_=len(history)-1
    for selection,cp in chosen.items():
        row=history[cp[0].grid_index_];row['selected']=True
        if row['collapsed']:
            warnings.warn(f'JTT {selection} validation ทายข้างเดียว อ่าน accuracy และ rate ควบคู่')
    return chosen,chosen['average'][0].selection_metadata_['jtt_upweight'],history


def choose_dro(xt, y, xv, yv, vg, groups, erm, config, seed, method,
               inferred_val_groups=None, inferred_min_size=1, variant="cy", sample_weights=None):
    """ค้นกริดและ checkpoint แยกแต่ละเกณฑ์; primary ใช้ inferred groups พร้อม guard"""
    if inferred_val_groups is None:
        raise ValueError('ต้องระบุกลุ่ม validation ของวิธีนี้ ห้ามใช้ประชากรแทนโดยปริยาย')
    erm_accuracy=float(((predict_mlp(erm,xv)>=.5)==yv).mean())
    floor=erm_accuracy-config.selection_accuracy_tolerance
    chosen={};best_ranks={};best_indices={};history=[]
    for eta, adjustment, learning_rate, epochs in product(config.dro_eta_grid, config.dro_adjustment_grid,
                                                         config.dro_learning_rate_grid, config.dro_epoch_grid):
        candidate=replace(config,dro_eta=eta,dro_adjustment=adjustment)
        checkpoints,final=train_mlp(xt,y,xv,yv,vg,candidate,seed,groups=groups,
            initial_model=erm,epochs=epochs,learning_rate=learning_rate,
            inferred_val_groups=inferred_val_groups,accuracy_floor=floor,inferred_min_size=inferred_min_size, sample_weights=sample_weights)
        for selection,cp in checkpoints.items():
            model,epoch,score=cp;meta=model.selection_metadata_
            settings={'dro_variant':variant,'dro_eta':eta,'dro_adjustment':adjustment,'dro_learning_rate':learning_rate,
                      'dro_finetune_epochs':epochs}
            model.selection_metadata_={**meta,**settings}
            row={'seed':seed,'method':method,'dro_variant':variant,'selection':selection,'dro_eta':eta,'C':adjustment,
                 'validation_accuracy':meta['average'],'validation_selection_score':score,
                 'validation_inferred_worst_group':meta['inferred_worst_group'],
                 'validation_true_worst_group':meta['worst_group'],'selected_epoch':epoch,
                 'selection_metric':selection,'warm_start':'ERM-MLP average checkpoint',
                 'finetune_epochs':epochs,'learning_rate':learning_rate,
                 'n_dro_groups':len(np.unique(groups)),'n_validation_groups':len(np.unique(inferred_val_groups)),
                 'validation_group_source':('true intersection' if method=='GDRO-oracle' else 'inferred cluster') + (' x y' if variant.startswith('cy') else ''),
                 'inferred_min_size':inferred_min_size,'erm_validation_accuracy':erm_accuracy,
                 'accuracy_floor':floor,'accuracy_guard_passed':meta['accuracy_guard_passed'],
                 'parameter_l2_from_erm':meta['parameter_l2_from_erm'],
                 'group_weight_min':meta['group_weight_min'],'group_weight_max':meta['group_weight_max'],
                 **final.training_diagnostics_,'selected':False}
            history.append(row);rank=selection_rank(selection,meta,floor)
            if selection not in best_ranks or rank>best_ranks[selection]:
                best_ranks[selection]=rank;chosen[selection]=cp;best_indices[selection]=len(history)-1
    for index in best_indices.values(): history[index]['selected']=True
    return chosen,history[best_indices['inferred_worst_group']],history


def run_experiment(config,root='.'):
    root=Path(root).resolve(); output=root/'results';output.mkdir(exist_ok=True)
    start=time.perf_counter(); set_seed(config.seeds[0],config.threads)
    before={str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()}
    prior_env=json.loads((output/'environment.json').read_text()) if (output/'environment.json').exists() else {}
    env=environment(config); setup_tracking(root)
    rows=[];clusters=[];resources=[];audits=[];explanations=[];feature_tests=[];jtt_history=[];dro_history=[];demo_bundle=None
    pp_rows=[];pp_thresholds=[];pp_audits=[];pp_demo=None
    with ResourceMonitor() as overall:
        frame,y,s,stages=load_hmda(root/'data',config.state)
        write_profile(output,stages,y,s)
        env.update(eligible_source_rows=len(y),source_rows=int(stages.iloc[0]['n']))
        for seed in config.seeds:
            print(f'seed={seed}: HMDA split / audit / representation',flush=True)
            splits=split_hmda(frame,y,s,seed,config.max_rows)
            train,val,test=(splits[n] for n in ['train','val','test'])
            audit={'seed':seed,**{f'{n}_ids':p['ids'].tolist() for n,p in splits.items()},'fits':[]}
            def add_fit(stage,ids):
                assert set(ids).issubset(train['ids']) and set(ids).isdisjoint(np.r_[val['ids'],test['ids']])
                audit['fits'].append({'stage':stage,'fit_ids':np.asarray(ids).tolist()})
            with ResourceMonitor() as pr:
                guard=LeakageGuard().fit(train['frame'],train['y'])
                for part in splits.values(): part['x']=guard.transform(part['frame'])
                processor=HMDAPreprocessor(config.min_category_frequency,config.lei_top_n).fit(train['x'])
                transformed={n:processor.transform(p['x']) for n,p in splits.items()}
            xt,xv,xx=(transformed[n] for n in ['train','val','test'])
            add_fit('leakage_guard',guard.fit_ids_);add_fit('imputer_scaler_categories_top_lei',processor.fit_ids_)
            audit.update(features=guard.features_,column_audit=guard.audit_)
            # กลุ่มจริงใช้เลือก oracle และผลเสริม worst_group เท่านั้น
            vg=val['s'].intersection.to_numpy()
            with ResourceMonitor() as er:
                erm_checkpoints,_=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed)
            erm=erm_checkpoints['average'][0]
            with ResourceMonitor() as sr:
                scarf=train_scarf(train['x'],processor,config,seed)
                scarf_scaler=StandardScaler().fit(encode_scarf(scarf,xt))
                reps={'raw':transformed,'scarf':{n:scarf_scaler.transform(encode_scarf(scarf,x)).astype(np.float32) for n,x in transformed.items()}}
            add_fit('scarf',scarf.fit_ids_);add_fit('scarf_scaler',train['ids'])
            with ResourceMonitor() as gr:
                george_scaler=StandardScaler().fit(hidden(erm,xt))
                reps['GEORGE']={n:george_scaler.transform(hidden(erm,x)).astype(np.float32) for n,x in transformed.items()}
            add_fit('GEORGE_erm_hidden_scaler',train['ids'])
            for name,seconds,model in [('raw',pr.seconds,None),('scarf',sr.seconds,scarf.encoder),('GEORGE',er.seconds+gr.seconds,erm)]:
                resources.append({'seed':seed,'representation':name,'discovery_train_seconds':seconds,
                                  'peak_memory_mb':overall.process.memory_info().rss/1024**2,
                                  **(network_cost(model) if model is not None else {'parameters':0,'flops_per_row':0})})
            discoveries={}; selected={}
            for definition in ['cy','c']:
                for rep_name,rep in reps.items():
                    for k in config.ks:
                        with ResourceMonitor() as cr:
                            km=(ClassClusters().fit(rep['train'],train['y'],train['ids'],k,seed)
                                if definition=='cy' else GlobalClusters().fit(rep['train'],train['ids'],k,seed))
                            labels={'train':km.labels_,**{n:km.predict(rep[n]) for n in ['val','test']}}
                            scores=group_scores(rep['train'],labels['train'],train['s'],seed)
                            alignment=group_scores(rep['test'],labels['test'],test['s'],seed)
                        add_fit(f'{definition}_kmeans_{rep_name}_{k}',km.fit_ids_)
                        seconds=cr.seconds+(sr.seconds if rep_name=='scarf' else er.seconds+gr.seconds if rep_name=='GEORGE' else 0)
                        rec={'seed':seed,'representation':rep_name,'k':k,'group_definition':definition,
                             'silhouette':scores['silhouette'],'davies_bouldin':scores['davies_bouldin'],
                             'n_clusters':scores['n_clusters'],
                             **{key:value for key,value in alignment.items() if key.startswith(('ari_','nmi_','alignment_'))},
                             'discovery_train_seconds':seconds,
                             'assignment':'nearest center without y', 'cluster_fit_uses_y':definition=='cy'}
                        clusters.append(rec);discoveries[(definition,rep_name,k)]=(km,labels,rec,seconds)
                    selected[(definition,rep_name)]=max(config.ks,key=lambda k:discoveries[(definition,rep_name,k)][2]['silhouette'])
            # HBAC ใช้ OOF error ของ ERM-LGBM บน train ไม่ใช้ demographic
            with ResourceMonitor() as hr:
                error,oof=oof_errors(train,config,seed)
                hbac=fit_hbac(xt,error,train['ids'],seed,k=4)
                labels={'train':hbac.labels_,**{n:hbac.predict(transformed[n]) for n in ['val','test']}}
                hscore=group_scores(xt,labels['train'],train['s'],seed)
                alignment=group_scores(xx,labels['test'],test['s'],seed)
            audit['error_crossfits']=[oof];add_fit('HBAC_OOF',train['ids']);add_fit('HBAC',hbac.fit_ids_)
            hrec={'seed':seed,'representation':'HBAC','k':hbac.n_clusters_,'silhouette':hscore['silhouette'],
                  'davies_bouldin':hscore['davies_bouldin'],'n_clusters':hbac.n_clusters_,
                  **{a:b for a,b in alignment.items() if a.startswith(('ari_','nmi_','alignment_'))},'discovery_train_seconds':hr.seconds}
            hlabels=labels
            clusters.append({**hrec,'group_definition':'shared'})
            resources.append({'seed':seed,'representation':'HBAC','discovery_train_seconds':hr.seconds,
                              'peak_memory_mb':hr.peak_mb,'parameters':np.nan,'flops_per_row':np.nan})
            with ResourceMonitor() as geo_r:
                geo_values={n:pd.to_numeric(p['frame'].tract_minority_population_percent,errors='coerce').to_numpy() for n,p in splits.items()}
                geo=GeoGroups().fit(geo_values['train'],train['ids'])
                geo_labels={n:geo.predict(v) for n,v in geo_values.items()}
                geo_score=group_scores(xx,geo_labels['test'],test['s'],seed)
            add_fit('geoproxy_quantiles',geo.fit_ids_)
            clusters.append({'seed':seed,'representation':'geoproxy','k':4,**geo_score,'discovery_train_seconds':geo_r.seconds})
            # ใช้ Silhouette train เลือก representation/k แยก c และ cy ก่อนเลือก variant ด้วย validation
            main_pairs={d:max([(r,selected[(d,r)]) for r in ['raw','scarf']],
                             key=lambda pair:discoveries[(d,*pair)][2]['silhouette']) for d in ['cy','c']}
            for (definition,rep_name),k in selected.items():
                _,labels,_,_=discoveries[(definition,rep_name,k)]
                method=f'GDRO-{rep_name}-k{k}' if rep_name in ['raw','scarf'] else rep_name
                cards,tests,fit_ids=describe_groups(method,k,seed,train,val,test,labels,predict_mlp(erm,xx),config)
                for row in cards+tests: row['group_definition']=definition
                explanations.extend(cards);feature_tests.extend(tests);add_fit(f'explanation_{definition}_{rep_name}',fit_ids)
            cards,tests,fit_ids=describe_groups('HBAC',hbac.n_clusters_,seed,train,val,test,hlabels,predict_mlp(erm,xx),config)
            for definition in ['cy','c']:
                explanations.extend([{**r,'group_definition':definition} for r in cards])
                feature_tests.extend([{**r,'group_definition':definition} for r in tests])
            add_fit('explanation_HBAC',fit_ids)
            def record(method,rep,k,checkpoints,seconds,peak,alignment=None,discovery_seconds=0,extra=None):
                for selection,(model,epoch,score) in checkpoints.items():
                    predict=(lambda a,model=model:predict_mlp(model,a)) if method!='ERM-LGBM' else (lambda a,epoch=epoch:model.predict_proba(a,num_iteration=epoch)[:,1])
                    row={'seed':seed,'method':method,'representation':rep,'k':k,'label_fraction':1.,'selection':selection,
                         'selected_epoch':epoch,'validation_selection_score':score,'n_train':len(xt),'n_val':len(xv),'n_test':len(xx),
                         'input_dimension':xt.shape[1],'threshold':.5,'train_seconds':pr.seconds+seconds+discovery_seconds,
                         'decision_train_seconds':seconds,'discovery_train_seconds':discovery_seconds,'peak_memory_mb':max(peak,pr.peak_mb),
                         'dro_variant':'none','group_definition':'none',
                         'selected_without_demographics':False,'proposed_selected':False,
                         'variant_selected_without_demographics':False,'proposed_variant_selected':False,**(alignment or {}),**(extra or {})}
                    if hasattr(model,'selection_metadata_'):
                        meta=model.selection_metadata_
                        row.update({key:value for key,value in meta.items() if key.startswith(('dro_','jtt_'))})
                        row.update(validation_average_accuracy=meta['average'],accuracy_guard_passed=meta['accuracy_guard_passed'],
                                   validation_accuracy_floor=meta['accuracy_floor'],parameter_l2_from_erm=meta['parameter_l2_from_erm'])
                    row.update(evaluate(test['y'],predict(xx),test['s'],config.min_fair_group))
                    row['inference_seconds_per_1000']=inference_seconds(predict,xx)
                    if method=='ERM-LGBM':
                        trees=model.booster_.dump_model(num_iteration=epoch)['tree_info']
                        row.update(parameters=np.nan,flops_per_row=np.nan,tree_count=len(trees),leaf_count=sum(t['num_leaves'] for t in trees))
                    else: row.update(network_cost(model))
                    rows.append(row)
            record('ERM-MLP','none',0,erm_checkpoints,er.seconds,er.peak_mb)
            with ResourceMonitor() as lr:
                lgb=LGBMClassifier(n_estimators=config.trees,num_leaves=15,max_depth=5,learning_rate=.05,verbosity=-1,
                    n_jobs=config.threads,random_state=seed,deterministic=True,force_col_wise=True)
                lgb.fit(xt,train['y'])
                choices={t:validation_scores(val['y'],lgb.predict_proba(xv,num_iteration=t)[:,1],vg) for t in sorted(set([10,25,50,config.trees]))}
                lc={sel:(lgb,t,choices[t][sel]) for sel in ['average','worst_group'] for t in [max(choices,key=lambda t:choices[t][sel])]}
            record('ERM-LGBM','none',0,lc,lr.seconds,lr.peak_mb)
            # PP ใช้ c เท่านั้น; เลือก GEORGE k ด้วย silhouette train เดิม
            pp_methods=[]
            for (definition,rep,k),(km,labels,rec,seconds) in discoveries.items():
                if definition!='c' or (rep=='GEORGE' and k!=selected[('c',rep)]): continue
                name=f'PP-{rep}-k{k}' if rep in ['raw','scarf'] else 'PP-GEORGE'
                pp_methods.append((name,rep,k,labels))
            for k in config.ks:
                random_model=XRandomGroups().fit(xt,train['ids'],k,seed)
                random_labels={n:random_model.predict(x) for n,x in transformed.items()}
                add_fit(f'PP_random_xhash_{k}',random_model.fit_ids_)
                pp_methods.append((f'PP-random-k{k}','random',k,random_labels))
            pp_methods.extend([('PP-HBAC','HBAC',hbac.n_clusters_,hlabels),
                ('PP-geoproxy','geoproxy',len(np.unique(geo_labels['train'])),geo_labels),
                ('PP-oracle','oracle',train['s'].intersection.nunique(),
                 {n:p['s'].intersection.to_numpy() for n,p in splits.items()})])
            for base in ['ERM-LGBM','ERM-MLP']:
                print(f'  {seed}: postprocess {base}',flush=True)
                if base=='ERM-LGBM':
                    iteration=lc['average'][1]
                    vp=lgb.predict_proba(xv,num_iteration=iteration)[:,1]
                else:
                    vp=predict_mlp(erm,xv)
                # ตัวหาเกณฑ์ไม่รับ test แม้แต่คะแนน เพื่อป้องกันเลือกจาก holdout
                candidates=fit_candidates(base,vp,val,pp_methods,config,seed)
                for meta,model,_ in candidates:
                    assert set(model.fit_ids_)==set(val['ids'])
                    assert set(model.fit_ids_).isdisjoint(test['ids'])
                pp_audits.append({'seed':seed,'base':base,'fit_split':'validation',
                    'threshold_fit_ids':val['ids'].tolist(),'test_ids':test['ids'].tolist(),
                    'candidate_count':len(candidates),'selection_uses_demographics':False,
                    'oracle_excluded_from_selection':True})
                tp=lgb.predict_proba(xx,num_iteration=iteration)[:,1] if base=='ERM-LGBM' else predict_mlp(erm,xx)
                pp_rows.append({'seed':seed,'base':base,'method':base,'representation':'none','k':0,
                    'objective':'baseline','fit_split':'none','proposed_selected':False,
                    'selected_without_demographics':False,'threshold_fit_seconds':0.,
                    **evaluate(test['y'],tp,test['s'],config.min_fair_group)})
                evaluated,thresholds=evaluate_candidates(candidates,tp,test,config.min_fair_group)
                pp_rows.extend(evaluated);pp_thresholds.extend(thresholds)
                for meta,model,labels in candidates:
                    if seed==config.seeds[0] and base=='ERM-LGBM' and meta['proposed_selected']:
                        rep,k=meta['representation'],meta['k']
                        pp_demo={'base':base,'base_model':lgb,'iteration':lc['average'][1],
                            'threshold_model':model,'method':meta['method'],'objective':meta['objective'],
                            'representation':rep,'clusterer':discoveries[('c',rep,k)][0]}
            pd.DataFrame(pp_rows).to_csv(output/'postprocess_results.csv',index=False)
            pd.DataFrame(pp_thresholds).to_csv(output/'postprocess_thresholds.csv',index=False)
            save_json(output/'postprocess_audit.json',pp_audits)
            with ResourceMonitor() as jr:
                jc,jw,jhistory=choose_jtt(xt,train,xv,val,vg,config,seed,erm_checkpoints['average'][2])
            jtt_history.extend(jhistory);record('JTT','none',0,jc,jr.seconds,jr.peak_mb,extra={'jtt_upweight':jw})
            methods=[]
            for variant in config.dro_variants:
                definition='cy' if variant.startswith('cy') else 'c'
                for k in config.ks:
                    # cy คงกลุ่มสุ่มต่อคลาสเดิม; c สุ่มโดยไม่รับ y
                    random=np.random.default_rng(seed+k).integers(k,size=len(xt))
                    random_val=np.random.default_rng(seed+k+10000).integers(k,size=len(xv))
                    if definition=='cy':
                        random=train['y']*k+random
                        random_val=val['y']*k+random_val
                    methods.append((variant,f'GDRO-random-k{k}','random',k,random,random_val,{},0))
                for (d,rep,k),(_,labels,rec,seconds) in discoveries.items():
                    if d!=definition or (rep=='GEORGE' and k!=selected[(d,rep)]): continue
                    name=f'GDRO-{rep}-k{k}' if rep in ['raw','scarf'] else rep
                    methods.append((variant,name,rep,k,labels['train'],labels['val'],rec,seconds))
                methods.append((variant,'HBAC','HBAC',hbac.n_clusters_,hlabels['train'],hlabels['val'],hrec,hr.seconds))
                methods.append((variant,'GDRO-geoproxy','geoproxy',4,geo_labels['train'],geo_labels['val'],geo_score,geo_r.seconds))
                _,oracle=np.unique(train['s'].intersection.to_numpy(),return_inverse=True)
                methods.append((variant,'GDRO-oracle','oracle',len(np.unique(oracle)),oracle,vg,{},0))
            audit['dro_merges']=[];variant_candidates={};demo_candidates={}
            for variant,method,rep,k,base_groups,val_clusters,alignment,seconds in methods:
                definition='cy' if variant.startswith('cy') else 'c'
                print(f'  {seed}: {method} / {variant}',flush=True)
                with ResourceMonitor() as mr:
                    merge_space=reps[rep]['train'] if rep in reps else xt
                    groups,merge=make_dro_groups(merge_space,base_groups,config.min_dro_group,definition,
                                                 train['y'] if definition=='cy' else None)
                    weights=balanced_cluster_weights(base_groups if definition=='cy' else groups,train['y']) if variant.endswith('_bal') else None
                    inferred_val=validation_groups(val_clusters,val['y'],definition)
                    audit['dro_merges'].append({'method':method,'dro_variant':variant,**merge,
                        'class_weight_split':'train' if weights is not None else 'none',
                        'validation_group_counts':{str(g):int((inferred_val==g).sum()) for g in np.unique(inferred_val)}})
                    add_fit(f'dro_merge_{method}_{variant}',train['ids'])
                    checkpoints,chosen,history=choose_dro(xt,train['y'],xv,val['y'],vg,groups,erm,config,seed,method,
                        inferred_val_groups=inferred_val,inferred_min_size=config.min_fair_group if rep=='oracle' else 1,
                        variant=variant,sample_weights=weights)
                dro_history.extend(history)
                metrics={a:b for a,b in alignment.items() if a.startswith(('ari_','nmi_','silhouette','davies_bouldin','n_clusters'))}
                is_proposed=rep in ['raw','scarf'] and (rep,k)==main_pairs[definition]
                record(method,rep,k,checkpoints,mr.seconds+er.seconds,mr.peak_mb,metrics,seconds,
                    extra={'n_dro_groups':len(np.unique(groups)),'n_dro_groups_before_merge':merge['n_groups_before'],
                           'dro_warm_start':True,'min_dro_group':config.min_dro_group,'dro_variant':variant,
                           'group_definition':definition,'class_balanced':weights is not None,
                           'class_weight_min':float(weights.min()) if weights is not None else 1.,
                           'class_weight_max':float(weights.max()) if weights is not None else 1.,
                           'selected_without_demographics':selected.get((definition,rep))==k or rep in ['HBAC','geoproxy'],
                           'proposed_selected':is_proposed})
                cp=checkpoints['inferred_worst_group'][0]
                variant_candidates[(method,variant)]=cp.selection_metadata_
                if seed==config.seeds[0] and is_proposed:
                    demo_candidates[variant]={'guard':guard,'processor':processor,'erm':erm,'fair':cp,
                        'selection':'inferred_worst_group','dro_variant':variant,'group_definition':definition,
                        'representation':rep,'scarf':scarf,'scarf_scaler':scarf_scaler,'clusterer':discoveries[(definition,rep,k)][0],
                        'defaults':{c:float(train['x'][c].median()) if pd.api.types.is_numeric_dtype(train['x'][c]) else train['x'][c].mode().iloc[0] for c in train['x']},
                        'method':method,'seed':seed}
            # การเปรียบเทียบนี้อ่านเฉพาะ validation metadata ไม่อ่านตัวชี้วัด test ที่ record แล้ว
            def variant_rank(key):
                meta=variant_candidates[key]
                return selection_rank('inferred_worst_group',meta,meta['accuracy_floor'])
            best_variants={method:max([key for key in variant_candidates if key[0]==method],key=variant_rank)[1]
                           for method,_ in variant_candidates}
            proposed_keys=[(f'GDRO-{main_pairs["cy" if v.startswith("cy") else "c"][0]}-k{main_pairs["cy" if v.startswith("cy") else "c"][1]}',v)
                           for v in config.dro_variants]
            best_proposed=max(proposed_keys,key=variant_rank)
            for row in rows:
                if row['seed']!=seed or row['dro_variant']=='none': continue
                row['variant_selected_without_demographics']=row['representation']!='oracle' and row['dro_variant']==best_variants[row['method']]
                row['proposed_variant_selected']=(row['method'],row['dro_variant'])==best_proposed
            if seed==config.seeds[0]: demo_bundle=demo_candidates[best_proposed[1]]
            for row in rows:
                if row['seed']==seed: row['mlflow_run_id']=log_row(row,env['libraries'],config)
            for row in pp_rows:
                if row['seed']==seed: row['mlflow_run_id']=log_row(row,env['libraries'],config,kind='postprocess')
            pd.DataFrame(pp_rows).to_csv(output/'postprocess_results.csv',index=False)
            audits.append(audit)
            pd.DataFrame(rows).to_csv(output/'hmda_pilot_results.csv',index=False)
            pd.DataFrame(clusters).to_csv(output/'clustering_results.csv',index=False)
            pd.DataFrame(dro_history).to_csv(output/'dro_grid.csv',index=False)
            save_json(output/'leakage_audit.json',{'scope':'train-only per seed','seeds':audits})
        export_explanations(output,explanations,feature_tests)
        pd.DataFrame(resources).to_csv(output/'representation_resources.csv',index=False)
        pd.DataFrame(jtt_history).to_csv(output/'jtt_validation_grid.csv',index=False)
        from .demo import save_bundle
        demo_bundle['postprocess']=pp_demo
        save_bundle(output,demo_bundle)
        from .hmda_reporting import generate_reports
        generate_reports(output)
    env.update(wall_seconds=time.perf_counter()-start,peak_process_rss_mb=overall.peak_mb,decision_rows=len(rows),
               clustering_rows=len(clusters),completed=True,colab_runtime_verified=False,hbac_backend='unsupervised-bias-detection 0.2.6')
    env.update(postprocess_rows=len(pp_rows),postprocess_fit_seconds=sum(r['threshold_fit_seconds'] for r in pp_rows),
               postprocess_accuracy_tolerance=config.selection_accuracy_tolerance,
               postprocess_search='parity interval bisection 20 steps; EO coordinate descent 10 sweeps, two starts')
    after={str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()}
    env.update(new_files=len(after-before),total_files=len(after),new_file_limit=200,
               max_new_files_completed_runs=max(prior_env.get('max_new_files_completed_runs',prior_env.get('new_files',0)),len(after-before)))
    pd.DataFrame([{k:env[k] for k in ['wall_seconds','peak_process_rss_mb','decision_rows','clustering_rows','new_files','total_files','max_new_files_completed_runs']}]).to_csv(output/'runtime_summary.csv',index=False)
    assert env['new_files']<=200,env['new_files']
    save_json(output/'environment.json',env)
    print(f"DONE: {len(rows)} decision rows; {env['wall_seconds']:.2f} s; peak RSS {overall.peak_mb:.1f} MiB; new files {env['new_files']}",flush=True)
    return env


In [ ]:
%%writefile src/fairgroups/tracking.py
"""SQLite เก็บ params/metrics/tags โดยไม่มี artifact รายรัน"""
from pathlib import Path
import numpy as np
import mlflow


def setup_tracking(root):
    import os
    os.environ.setdefault('MLFLOW_ENABLE_TELEMETRY', 'false')
    root = Path(root).resolve()
    mlflow.set_tracking_uri('sqlite:///' + str(root / 'mlflow.db'))
    client = mlflow.tracking.MlflowClient()
    name = 'hmda-hidden-groups-pilot'
    exp = client.get_experiment_by_name(name)
    if exp is None:
        exp_id = client.create_experiment(name, artifact_location=(root/'results'/'no_run_artifacts').as_uri())
    else:
        exp_id = exp.experiment_id
    mlflow.set_experiment(experiment_id=exp_id)


def log_row(row, versions, config, kind='decision'):
    keys = ['method','representation','k','label_fraction','seed','selection','selected_epoch','n_train','n_test',
            'dro_variant','group_definition','class_balanced','base','objective','fit_split']
    with mlflow.start_run(run_name=f"{kind}-{row.get('method', row.get('representation','none'))}-{row['seed']}") as run:
        mlflow.set_tags({'kind':kind,'dataset':config.dataset,'state':config.state,'artifact_policy':'none'})
        mlflow.log_params({k:row[k] for k in keys if k in row})
        mlflow.log_params({f'version_{k}':v for k,v in versions.items()})
        mlflow.log_params({f'config_{k}':str(v) for k,v in config.to_dict().items()})
        mlflow.log_metrics({k:float(v) for k,v in row.items() if isinstance(v,(int,float,np.number)) and np.isfinite(v)})
        return run.info.run_id


In [ ]:
%%writefile tests/conftest.py
"""ให้ pytest หาโมดูลใน src ได้โดยไม่ต้องติดตั้งโปรเจกต์"""
import sys
import os
from pathlib import Path
if sys.platform == "darwin":
    os.environ["OMP_NUM_THREADS"] = "1"
ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
%%writefile tests/test_groupdef.py
"""ตรวจนิยามกลุ่มและสูตร class balance ที่มีผลต่อ ablation รอบ 5"""
import inspect
import numpy as np
import torch
import pytest
from fairgroups.dro import balanced_cluster_weights, make_dro_groups, validation_groups, group_dro_loss
from fairgroups.hmda_groups import GlobalClusters


def test_balanced_weights_equal_class_mass_within_each_cluster():
    c=np.array([0]*10+[1]*8+[2]*3)
    y=np.array([0]*2+[1]*8+[0]*6+[1]*2+[1]*3)
    weights=balanced_cluster_weights(c,y)
    for cluster in np.unique(c):
        mask=c==cluster
        np.testing.assert_allclose(weights[mask].mean(),1.)
        totals=[weights[mask & (y==label)].sum() for label in np.unique(y[mask])]
        np.testing.assert_allclose(totals,totals[0])
    np.testing.assert_array_equal(weights[c==2],np.ones(3))


def test_c_clustering_merging_and_validation_do_not_use_y():
    rng=np.random.default_rng(3);x=rng.normal(size=(80,3)).astype(np.float32)
    model=GlobalClusters().fit(x,np.arange(80),4,17)
    assert 'y' not in inspect.signature(GlobalClusters.fit).parameters
    base=model.labels_.copy()
    groups,audit=make_dro_groups(x,base,20,'c')
    changed,other=make_dro_groups(x,base,20,'c',y=rng.integers(2,size=80))
    np.testing.assert_array_equal(groups,changed);assert audit==other
    assert all('y' not in row for row in audit['mapping'])
    a=validation_groups(base,np.zeros(80),'c');b=validation_groups(base,np.ones(80),'c')
    np.testing.assert_array_equal(a,b)
    before=model.centers_.copy();model.predict(x*100)
    np.testing.assert_array_equal(model.centers_,before)
    assert set(model.fit_ids_)==set(range(80))


def test_cy_requires_y_and_crosses_oracle_intersections():
    c=np.array(['A','A','B','Unknown']);y=np.array([0,1,0,1])
    assert validation_groups(c,y,'cy').tolist()==['A|y=0','A|y=1','B|y=0','Unknown']
    assert validation_groups(c,y,'c').tolist()==c.tolist()
    with pytest.raises(ValueError): validation_groups(c,definition='cy')
    with pytest.raises(ValueError): make_dro_groups(np.zeros((4,1)),np.arange(4),definition='cy')


def test_balanced_c_loss_and_gradients_use_train_weights():
    c=np.zeros(4,dtype=int);y=np.array([0,1,1,1])
    weights=torch.tensor(balanced_cluster_weights(c,y))
    losses=torch.tensor([6.,3.,3.,3.],requires_grad=True)
    loss=group_dro_loss(losses*weights,torch.tensor(c),torch.zeros(1),0.)
    torch.testing.assert_close(loss,torch.tensor(4.5))
    loss.backward();torch.testing.assert_close(losses.grad,torch.tensor([.5,1/6,1/6,1/6]))


def test_cy_balanced_scales_pure_label_dro_losses_in_mixed_parent_cluster():
    y=np.array([0,1,1,1]);weights=torch.tensor(balanced_cluster_weights(np.zeros(4),y))
    losses=torch.ones(4,requires_grad=True)
    loss=group_dro_loss(losses*weights,torch.tensor(y),torch.zeros(2),0.)
    # เฉลี่ยกลุ่มย่อยหลังถ่วง (2 + 2/3)/2; หาร sum(w) ในกลุ่มจะทำให้เหลือ 1 ผิดนิยาม
    torch.testing.assert_close(loss,torch.tensor(4/3))
    loss.backward();torch.testing.assert_close(losses.grad,torch.tensor([1.,1/9,1/9,1/9]))


def test_single_class_parent_cluster_balance_is_explicit_noop():
    y=np.array([0,0,1,1,1])
    np.testing.assert_array_equal(balanced_cluster_weights(np.array([0,0,1,1,1]),y),np.ones(5))


def test_oracle_sanity_keeps_four_independent_variant_outcomes():
    import pandas as pd
    from fairgroups.hmda_reporting import oracle_sanity
    rows=[]
    for seed in [17,29]:
        rows.append({'seed':seed,'method':'ERM-MLP','selection':'average','dro_variant':'none','worst_group_accuracy':.8})
        for variant,value in [('cy',.7),('cy_bal',.75),('c',.82),('c_bal',.79)]:
            rows.append({'seed':seed,'method':'GDRO-oracle','selection':'inferred_worst_group',
                         'dro_variant':variant,'worst_group_accuracy':value})
    sanity,status,gains=oracle_sanity(pd.DataFrame(rows))
    assert len(sanity)==8 and status=={'cy':False,'cy_bal':False,'c':True,'c_bal':False}
    assert gains['c']>0 and gains['c_bal']<0


In [ ]:
%%writefile tests/test_hmda.py
"""ทดสอบเส้นทางรั่วและพฤติกรรม HMDA ที่มีผลต่อข้อสรุป"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pytest
from config import Config
from fairgroups.hmda import (LeakageGuard,HMDAPreprocessor,POST,FEATURES,prohibited,
                             demographics,split_hmda,dti_numeric)
from fairgroups.hmda_groups import ClassClusters,fit_hbac
from fairgroups.run import choose_jtt,oof_errors
from fairgroups.models import predict_mlp
from fairgroups.metrics import evaluate

ROOT=Path(__file__).resolve().parents[1]


def test_all_post_decision_and_sensitive_features_rejected():
    y=np.tile([0,1],40)
    cols=FEATURES+list(POST)+['denial_reason-1','tract_minority_population_percent',
        'applicant_race-1','co-applicant_ethnicity-1','applicant_sex_observed',
        'applicant_age_above_62','derived_race','derived_ethnicity','derived_sex']
    x=pd.DataFrame({c:np.ones(80) for c in cols})
    guard=LeakageGuard().fit(x,y)
    assert set(guard.features_)==set(FEATURES)
    assert all(not prohibited(c) for c in guard.features_)
    for c in cols:
        if prohibited(c):
            with pytest.raises(AssertionError): HMDAPreprocessor().fit(pd.DataFrame({c:np.ones(80)}))


def test_missing_gap_train_only_and_dti_order():
    y=np.tile([0,1],40)
    x=pd.DataFrame({'loan_amount':np.arange(80,dtype=float),'income':np.where(y==0,'Exempt','40')})
    guard=LeakageGuard().fit(x,y)
    assert guard.features_==['loan_amount']
    assert next(r for r in guard.audit_ if r['column']=='income')['missing_gap']==1
    before=guard.features_.copy();guard.transform(x.assign(income='99',loan_amount=np.nan))
    assert guard.features_==before
    np.testing.assert_array_equal(dti_numeric(pd.Series(['<20%','20%-<30%','30%-<36%','36','50%-60%','>60%'])),[10,25,33,36,55,65])


def test_unknown_retained_but_excluded_from_fairness():
    frame=pd.DataFrame({'derived_race':['White','Race Not Available','Race Not Available','Black or African American'],
                         'derived_ethnicity':['Not Hispanic or Latino','Hispanic or Latino','Ethnicity Not Available','Not Hispanic or Latino'],
                         'derived_sex':['Male','Female','Sex Not Available','Joint'],'applicant_age':['35-44','<25','8888','>74']})
    s=demographics(frame)
    assert s.race_eth.tolist()==['NH White','Hispanic','Unknown','NH Black']
    assert len(s)==4
    m=evaluate(np.array([1,0,0,1]),np.array([.9,.1,.8,.8]),s,min_detail=1)
    assert m['excluded_unknown_race_eth']==1 and m['excluded_unknown_sex']==1
    assert 'Unknown' not in json.loads(m['group_accuracy_intersection'])


def test_hmda_unsupervised_fits_never_include_holdout():
    rng=np.random.default_rng(17);n=160
    frame=pd.DataFrame({'loan_amount':rng.normal(size=n),'income':rng.normal(size=n),'lei':np.tile(['a','b','c','d'],40)},index=np.arange(500,500+n))
    y=np.tile([0,1],80)
    s=pd.DataFrame({'race_eth':np.tile(['NH White','NH Black'],80),'sex':'Male','age':'35-44','intersection':np.tile(['a','b'],80)},index=frame.index)
    splits=split_hmda(frame,y,s,17)
    tr=splits['train'];guard=LeakageGuard().fit(tr['frame'],tr['y']);tr['x']=guard.transform(tr['frame'])
    pre=HMDAPreprocessor(2,lei_top_n=1).fit(tr['x']);x=pre.transform(tr['x'])
    km=ClassClusters().fit(x,tr['y'],tr['ids'],2,17)
    saved=km.centers_.copy();pre.transform(guard.transform(splits['test']['frame'].assign(income=99999)))
    km.predict(x*100);np.testing.assert_array_equal(km.centers_,saved)
    for obj in [guard,pre,km]:
        assert set(obj.fit_ids_)==set(tr['ids'])
        assert set(obj.fit_ids_).isdisjoint(np.r_[splits['test']['ids'],splits['val']['ids']])
    error,audit=oof_errors(tr,Config(oof_folds=2,trees=3),17)
    held=[]
    for f in audit['folds']:
        assert set(f['fit_ids']).isdisjoint(f['heldout_ids'])
        assert set(f['guard_fit_ids'])==set(f['preprocessor_fit_ids'])==set(f['fit_ids'])
        held.extend(f['heldout_ids'])
    assert sorted(held)==sorted(tr['ids']) and np.isfinite(error).all()


def test_hbac_package_assigns_new_rows_without_bias_metric():
    rng=np.random.default_rng(2);x=np.r_[rng.normal(-3,.1,(40,3)),rng.normal(3,.1,(40,3))].astype(np.float32)
    m=fit_hbac(x,np.r_[np.zeros(40),np.ones(40)],np.arange(80),17,k=2)
    assert m.n_clusters_==2
    np.testing.assert_array_equal(m.predict(x),m.labels_)


def test_jtt_validation_grid_does_not_collapse_on_separable_data():
    rng=np.random.default_rng(7);x=rng.normal(size=(400,4)).astype(np.float32);y=(x[:,0]>0).astype(int)
    cfg=Config(epochs=12,jtt_identification_epochs=3,hidden=16,batch_size=32)
    cp,weight,history=choose_jtt(x[:300],{'y':y[:300]},x[300:],{'y':y[300:]},np.zeros(100),cfg,17)
    p=predict_mlp(cp['average'][0],x[300:]);assert .05<(p>=.5).mean()<.95
    assert weight==max([r for r in history if r['selection']=='average'],key=lambda r:r['validation_accuracy'])['upweight']


def test_hmda_recorded_audit_and_jtt():
    path=ROOT/'results/leakage_audit.json'
    if not path.exists(): pytest.skip('ยังไม่มีผล HMDA จริง')
    audits=json.loads(path.read_text())['seeds']
    for a in audits:
        train,val,test=(set(a[f'{n}_ids']) for n in ['train','val','test'])
        assert train.isdisjoint(val|test) and val.isdisjoint(test)
        assert all(not prohibited(c) for c in a['features'])
        for f in a['fits']:
            assert set(f['fit_ids']).issubset(train) and set(f['fit_ids']).isdisjoint(val|test)
        for oof in a['error_crossfits']:
            held=[]
            for f in oof['folds']:
                assert set(f['fit_ids']).isdisjoint(f['heldout_ids'])
                assert set(f['fit_ids'])|set(f['heldout_ids'])==train
                assert set(f['guard_fit_ids'])==set(f['preprocessor_fit_ids'])==set(f['fit_ids'])
                held.extend(f['heldout_ids'])
            assert sorted(held)==sorted(train)
        for r in a['column_audit']:
            if not r['dropped']: assert r['missing_gap']<=.10
    results=pd.read_csv(ROOT/'results/hmda_pilot_results.csv')
    jtt=results[(results.method=='JTT') & (results.selection=='average')]
    # ถ้าโมเดลยุบเป็นคลาสเดียว งานจริงต้อง fail ให้แก้หรือรายงาน ไม่ซ่อนด้วย EO
    assert jtt.positive_rate.between(.005,.995,inclusive='neither').all()


def test_demo_build_and_prediction():
    if not (ROOT/'results/demo_bundle.joblib').exists(): pytest.skip('ยังไม่มี bundle จาก pilot')
    from fairgroups.demo import load_bundle,predict_application,build_interface
    b=load_bundle(ROOT/'results');values=[b['defaults'][c] for c in b['inputs']]
    result,card=predict_application(b,*values)
    assert 0<=result['โอกาสอนุมัติ ERM']<=1
    assert 0<=result['โอกาสอนุมัติหลังฝึก DRO']<=1
    assert 'วิจัย' in card and str(result['กลุ่ม']) in card
    assert len(b['inputs'])==8
    demo=build_interface(ROOT/'results');assert callable(demo.fn)
    second=demo.fn(*values);assert second[0]==result


In [ ]:
%%writefile tests/test_leakage.py
"""ตรวจการรั่วจากเลขแถวจริงและจากข้อมูลจำลองที่มีค่า holdout แปลกออกไป"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pytest
from config import Config
from fairgroups.data import (TrainPreprocessor, FEATURES, NUMERIC, SENSITIVE,
                             split_data, labeled_indices)
from fairgroups.representations import crossfit_error, train_scarf
from fairgroups.clustering import discover


def sample_data():
    rng = np.random.default_rng(12)
    n = 160
    x = pd.DataFrame({c: rng.integers(1, 6, n).astype(float) for c in FEATURES},
                     index=np.arange(1000, 1000+n))
    y = np.tile([0, 1], n // 2)
    s = pd.DataFrame({"SEX": np.tile([1, 1, 2, 2], n//4),
                      "RAC1P": np.tile([1, 1, 1, 1, 2, 2, 2, 2], n//8)}, index=x.index)
    return x, y, s


def test_train_only_scaler_encoder_scarf_kmeans():
    x, y, s = sample_data()
    split = split_data(x, y, s, seed=17)
    train, test = split["train"], split["test"]
    processor = TrainPreprocessor(2).fit(train["x"])
    test_changed = test["x"].copy()
    test_changed.loc[:, "AGEP"] = 99999
    processor.transform(test_changed)
    assert not SENSITIVE.intersection(processor.feature_names_in_)
    assert set(processor.fit_ids_).isdisjoint(test["ids"])
    expected_mean = train["x"][NUMERIC].mean().to_numpy()
    np.testing.assert_allclose(processor.encoder.named_transformers_["numeric"].mean_, expected_mean)
    config = Config(scarf_epochs=1, hidden=8, embedding=4, batch_size=32)
    scarf = train_scarf(train["x"], processor, config, 17)
    _, _, km = discover(processor.transform(train["x"]), train["ids"], train["s"], 2, 17)
    for estimator in [processor, scarf, km]:
        assert set(estimator.fit_ids_) == set(train["ids"])
        assert set(estimator.fit_ids_).isdisjoint(test["ids"])
        assert set(estimator.fit_ids_).isdisjoint(split["val"]["ids"])


def test_sensitive_features_are_rejected():
    x, _, _ = sample_data()
    for sensitive in SENSITIVE:
        with pytest.raises(AssertionError):
            TrainPreprocessor().fit(x.assign(**{sensitive: 1}))


def test_oof_exactly_once_within_label_budget():
    x, y, s = sample_data()
    part = split_data(x, y, s, seed=17)["train"]
    labeled = labeled_indices(part["y"], 0.5, 17)
    processor = TrainPreprocessor(2).fit(part["x"])
    ids = part["ids"][labeled]
    rep, audit = crossfit_error(processor.transform(part["x"])[labeled], part["y"][labeled],
                               ids, Config(trees=3), 17)
    assert rep.shape == (len(labeled), 3)
    held = []
    for fold in audit["folds"]:
        assert set(fold["fit_ids"]).isdisjoint(fold["heldout_ids"])
        assert set(fold["fit_ids"]) | set(fold["heldout_ids"]) == set(ids)
        held.extend(fold["heldout_ids"])
    assert sorted(held) == sorted(ids)
    assert len(ids) == len(part["y"]) // 2


def test_recorded_experiment_has_no_holdout_in_fit():
    path = Path(__file__).resolve().parents[1] / "results/acs_round1/leakage_audit.json"
    if not path.exists():
        pytest.skip("ยังไม่ได้รันชุดจริง; หลัง --quick จะตรวจหลักฐานนี้อีกครั้ง")
    for audit in json.loads(path.read_text()):
        train, val, test = (set(audit[f"{name}_ids"]) for name in ["train", "val", "test"])
        assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
        assert not SENSITIVE.intersection(audit["features"])
        for fit in audit["fits"]:
            assert set(fit["fit_ids"]).issubset(train)
            assert set(fit["fit_ids"]).isdisjoint(test | val)
        for error in audit["error_crossfits"]:
            labeled = set(error["fit_ids"])
            assert len(labeled) == int(len(train) * error["label_fraction"])
            held = []
            for fold in error["folds"]:
                assert set(fold["fit_ids"]).isdisjoint(fold["heldout_ids"])
                assert set(fold["fit_ids"]) | set(fold["heldout_ids"]) == labeled
                held.extend(fold["heldout_ids"])
            assert len(held) == len(set(held)) == len(labeled)


In [ ]:
%%writefile tests/test_methods.py
"""ตรวจสูตรที่มีผลต่อข้อสรุป ไม่ใช่เพียงตรวจว่าโค้ดเรียกได้"""
import numpy as np
import torch
from config import Config
from fairgroups.dro import group_dro_loss
from fairgroups.metrics import group_metrics
from fairgroups.models import train_mlp, predict_mlp
from fairgroups.efficiency import network_cost


def test_dro_averages_groups_not_examples():
    losses = torch.tensor([1., 1., 4.], requires_grad=True)
    groups = torch.tensor([0, 0, 1])
    value = group_dro_loss(losses, groups, torch.zeros(2), eta=0)
    assert float(value.detach()) == 2.5
    value.backward()
    np.testing.assert_allclose(losses.grad.numpy(), [0.25, 0.25, 0.5])


def test_known_fairness_values():
    y = np.array([1, 1, 0, 0, 1, 1, 0, 0])
    pred = np.array([1, 1, 0, 0, 1, 0, 1, 0])
    scores = group_metrics(y, pred, np.array(["a"]*4 + ["b"]*4))
    assert scores["eo_gap"] == 0.5
    assert scores["dp_difference"] == 0
    assert scores["worst_group_accuracy"] == 0.5


def test_average_selection_is_invariant_to_population_groups():
    rng = np.random.default_rng(1)
    x = rng.normal(size=(40, 4)).astype(np.float32)
    y = (x[:, 0] > 0).astype(int)
    cfg = Config(epochs=3, hidden=8, batch_size=16)
    a, _ = train_mlp(x, y, x, y, np.zeros(40), cfg, seed=17)
    b, _ = train_mlp(x, y, x, y, np.arange(40) % 4, cfg, seed=17)
    assert a["average"][1] == b["average"][1]
    np.testing.assert_array_equal(predict_mlp(a["average"][0], x), predict_mlp(b["average"][0], x))


def test_linear_flops_formula():
    model = torch.nn.Sequential(torch.nn.Linear(3, 4), torch.nn.ReLU(), torch.nn.Linear(4, 1))
    assert network_cost(model) == {"parameters": 21, "flops_per_row": 32}


In [ ]:
%%writefile tests/test_postprocess.py
"""ตรวจ validation-only, การอนุมานจาก x และ guard ด้วยข้อมูลที่ไม่สมดุล"""
import inspect
import itertools
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pytest
from config import Config
from fairgroups.postprocess import ValidationThresholds, XRandomGroups, fit_candidates, selection_key
from fairgroups.hmda_groups import GlobalClusters
from fairgroups.metrics import evaluate


def sample():
    rng=np.random.default_rng(19)
    groups=np.repeat([0,1,2], [100,100,12])
    y=(rng.random(len(groups))<np.where(groups==0,.95,.7)).astype(int)
    p=np.clip(.3+.5*y+rng.normal(0,.2,len(y)),.01,.99)
    return p,y,groups,np.arange(1000,1000+len(y))


@pytest.mark.parametrize('objective',['parity','eo'])
def test_validation_only_fit_and_frozen_small_unseen_groups(objective):
    p,y,g,ids=sample()
    with pytest.raises(ValueError,match='validation'):
        ValidationThresholds().fit(p,y,g,ids,split='test')
    model=ValidationThresholds(objective).fit(p,y,g,ids)
    assert model.fit_split_=='validation' and set(model.fit_ids_)==set(ids)
    assert model.thresholds_[2]==.5
    np.testing.assert_array_equal(model.predict([.49,.51],[999,999]),[0,1])
    assert model.validation_['accuracy']>=model.baseline_accuracy_-.02-1e-12
    frozen=model.thresholds_.copy()
    model.predict(np.array([.01,.99]),np.array([0,1]))
    assert frozen==model.thresholds_  # test prediction เปลี่ยนค่า fit ไม่ได้
    assert 'test' not in inspect.signature(fit_candidates).parameters


def test_parity_matches_bruteforce_grid_optimum_and_eo_improves_its_target():
    p,y,g,ids=sample()
    parity=ValidationThresholds().fit(p,y,g,ids)
    best=1.
    # ชุดนี้มี 2 กลุ่มที่ขยับได้ ใช้ brute force ยืนยันตัวค้นช่วง
    for i,j in itertools.product(range(len(parity.tables_[0]['threshold'])),range(len(parity.tables_[1]['threshold']))):
        s=parity._stats([i,j,0])
        if s['accuracy']>=parity.accuracy_floor_-1e-12:
            best=min(best,s['approval_gap'])
    assert parity.validation_['approval_gap']<=best+1e-6
    eo=ValidationThresholds('eo').fit(p,y,g,ids)
    assert eo.validation_['eo_gap']<=parity.validation_['eo_gap']+1e-12
    assert eo.validation_['eo_gap']<=eo._stats(np.zeros(3,dtype=int))['eo_gap']+1e-12


def test_proposed_and_random_assignment_are_x_only_stable_per_row():
    rng=np.random.default_rng(7);x=rng.normal(size=(80,3)).astype(np.float32)
    cluster=GlobalClusters().fit(x,np.arange(80),4,17)
    random=XRandomGroups().fit(x,np.arange(80),4,17)
    for model in [cluster,random]:
        assert set(inspect.signature(model.predict).parameters)=={'x'}
        before=model.predict(x)
        np.testing.assert_array_equal(model.predict(x[::-1])[::-1],before)
        np.testing.assert_array_equal(model.predict(x[:1]),before[:1])


def test_selection_ignores_true_group_and_test_metrics_and_auroc_uses_scores():
    p,y,g,ids=sample();val={'y':y,'ids':ids}
    methods=[('PP-raw-k2','raw',2,{'val':g}),('PP-oracle','oracle',3,{'val':g}),
             ('PP-random-k2','random',2,{'val':g})]
    candidates=fit_candidates('ERM-MLP',p,val,methods,Config(),17)
    selected=[r for r,_,_ in candidates if r['proposed_selected']]
    assert len(selected)==1 and selected[0]['representation']=='raw'
    row=selected[0]
    assert selection_key(row)==selection_key({**row,'test_accuracy':0,'approval_rate_gap_race_eth':999})
    sensitive=pd.DataFrame({'race_eth':np.where(g==0,'White','Non-white'),
        'sex':np.where(np.arange(len(y))%2,'Male','Female'),'age':np.repeat('25-44',len(y))})
    sensitive['intersection']=sensitive.race_eth+'|'+sensitive.sex
    baseline=evaluate(y,p,sensitive,1)
    changed=evaluate(y,p,sensitive,1,prediction=np.zeros(len(y),dtype=int))
    assert baseline['auroc']==changed['auroc'] and changed['positive_rate']==0


def test_completed_pp_audit_and_small_group_thresholds():
    output=Path(__file__).resolve().parents[1]/'results'
    if not (output/'postprocess_audit.json').exists():
        pytest.skip('ยังไม่มีผลรอบ 6')
    for audit in json.loads((output/'postprocess_audit.json').read_text()):
        assert audit['fit_split']=='validation'
        assert set(audit['threshold_fit_ids']).isdisjoint(audit['test_ids'])
        assert audit['candidate_count']==26
    thresholds=pd.read_csv(output/'postprocess_thresholds.csv')
    assert thresholds[thresholds.frozen_small_group].threshold.eq(.5).all()
    results=pd.read_csv(output/'postprocess_results.csv')
    for _,part in results.groupby(['base','seed']):
        assert part.proposed_selected.sum()==part.selected_without_demographics.sum()==1
        assert part[part.proposed_selected].representation.isin(['raw','scarf']).all()
        assert part.auroc.nunique()==1


In [ ]:
%%writefile tests/test_tracking.py
"""SQLite ต้องอ่านผลกลับได้ และไม่มี artifact รายรัน"""
import mlflow
from fairgroups.tracking import setup_tracking, log_row
from config import Config


def test_sqlite_tracking_has_no_per_run_artifacts(tmp_path):
    setup_tracking(tmp_path)
    run_id=log_row({'seed':17,'representation':'raw','method':'test','accuracy':.75},{},Config())
    fetched=mlflow.get_run(run_id)
    assert fetched.data.metrics['accuracy']==.75
    assert fetched.data.params['seed']=='17'
    assert fetched.info.status=='FINISHED'
    assert mlflow.get_tracking_uri().startswith('sqlite:///')
    assert not mlflow.tracking.MlflowClient().list_artifacts(run_id)
    assert len([p for p in tmp_path.rglob('*') if p.is_file() and not p.name.startswith('._')])<=3


In [ ]:
%%writefile tests/test_tuning.py
"""ตรวจการรวมกลุ่ม การถ่วง loss และคำอธิบายที่มีผลต่อข้อสรุปรอบ 3"""
import copy
import json
import numpy as np
import pandas as pd
import torch
from config import Config
from fairgroups.dro import merge_small_groups, group_dro_loss
from fairgroups.explanations import fit_surrogate, feature_differences, card_text, choose_surrogate
from fairgroups.models import train_mlp, predict_mlp, cross_labels, selection_rank, worst_accuracy
from fairgroups.run import choose_dro


def test_merge_chooses_nearest_center_only_within_class():
    x=np.r_[np.full(2,1.),np.zeros(8),np.full(8,10.),np.ones(8)][:,None]
    y=np.r_[np.zeros(18),np.ones(8)].astype(int)
    clusters=np.r_[np.zeros(2),np.ones(8),np.full(8,2),np.full(8,3)].astype(int)
    groups, details=merge_small_groups(x,y,clusters,min_size=5)
    assert np.all(groups[:2]==groups[2])
    assert groups[0]!=groups[-1] and groups[0]!=groups[10]
    assert details['n_groups_before']==4 and details['n_groups_after']==3
    assert min(details['group_sizes'])>=5
    for group in np.unique(groups):
        assert len(np.unique(y[groups==group]))==1


def test_merge_small_groups_together_and_report_undersized_class():
    x=np.arange(9,dtype=float)[:,None];y=np.r_[np.zeros(6),np.ones(3)]
    groups, details=merge_small_groups(x,y,np.arange(9),min_size=5)
    assert sorted(details['group_sizes'])==[3,6]
    assert details['undersized_remaining']==1
    np.testing.assert_array_equal(groups,merge_small_groups(x,y,np.arange(9),5)[0])


def test_adjustment_uses_full_train_counts_and_excludes_absent_batch_group():
    losses=torch.ones(3,requires_grad=True);log_weights=torch.zeros(3)
    value=group_dro_loss(losses,torch.tensor([0,0,1]),log_weights,eta=1,
                         group_counts=torch.tensor([100.,4.,1.]),adjustment=1)
    expected=torch.log_softmax(torch.tensor([1.1,1.5,0.]),dim=0)
    torch.testing.assert_close(log_weights,expected)
    value.backward()
    w=torch.softmax(torch.tensor([1.1,1.5]),dim=0)
    torch.testing.assert_close(losses.grad,torch.tensor([w[0]/2,w[0]/2,w[1]]))


def test_small_separable_surrogate_predicts_both_sides_on_holdout():
    train=np.r_[np.zeros(990),np.ones(10)][:,None]
    target=np.r_[np.zeros(990),np.ones(10)].astype(int)
    tree=fit_surrogate(train,target,17)
    holdout=np.r_[np.zeros(200),np.ones(10)][:,None]
    truth=np.r_[np.zeros(200),np.ones(10)].astype(int)
    np.testing.assert_array_equal(tree.predict(holdout),truth)
    assert tree.get_depth()<=3 and tree.class_weight=='balanced'


def test_fallback_effect_sizes_and_low_fidelity_card():
    frame=pd.DataFrame({'loan_amount':[0.,2.,2.,4.],'loan_type':['a','a','b','b']})
    differences=feature_differences(frame,np.array([True,True,False,False]))
    numeric=next(d for d in differences if d['feature']=='loan_amount')
    np.testing.assert_allclose(numeric['value'],-np.sqrt(2))
    category=next(d for d in differences if d['feature']=='loan_type')
    assert abs(category['value'])==1
    row={'population':'{}','method':'test','seed':17,'group':0,'rules':'[]',
         'n_train':10,'n_val':2,'n_test':2,'coverage':.1,'fidelity_f1':.2,
         'fidelity_precision':.25,'fidelity_recall':.5,'rule_length':3,'max_conditions_per_path':3,
         'approval_rate':.5,'erm_error':.5,'fpr':0.,'fnr':1.,
         'fallback_features':json.dumps(differences)}
    card=card_text(row)
    assert 'กลุ่มนี้อธิบายด้วยกฎลึก 3 ชั้นได้ไม่ดี' in card
    assert 'loan_amount' in card and 'loan_type' in card and 'train เท่านั้น' in card
    assert 'precision 0.2500' in card and 'recall 0.5000' in card


def test_dro_grid_selection_invariant_to_validation_demographics_and_preserves_erm():
    rng=np.random.default_rng(7);x=rng.normal(size=(120,3)).astype(np.float32)
    y=(x[:,0]>0).astype(int);cfg=Config(epochs=2,dro_epoch_grid=(2,),hidden=8,batch_size=32)
    cp,_=train_mlp(x[:80],y[:80],x[80:],y[80:],np.zeros(40),cfg,17)
    erm=cp['average'][0];saved=copy.deepcopy(erm.state_dict())
    a, selected, history=choose_dro(x[:80],y[:80],x[80:],y[80:],np.zeros(40),y[:80],erm,cfg,17,'test',inferred_val_groups=cross_labels((x[80:,1]>0).astype(int),y[80:]))
    b, other, _=choose_dro(x[:80],y[:80],x[80:],y[80:],np.arange(40)%4,y[:80],erm,cfg,17,'test',inferred_val_groups=cross_labels((x[80:,1]>0).astype(int),y[80:]))
    assert len(history)==24 and sum(r['selected'] for r in history)==3
    primary=[r for r in history if r['selection']=='inferred_worst_group']
    def rank(r):
        return selection_rank('inferred_worst_group',{'average':r['validation_accuracy'],
            'inferred_worst_group':r['validation_selection_score']},r['accuracy_floor'])
    assert selected==max(primary,key=rank)
    assert selected['parameter_l2_from_erm']>0
    assert all(r['final_group_weight_max']>r['final_group_weight_min'] for r in history)
    assert {k:v for k,v in selected.items() if k!='validation_true_worst_group'}=={k:v for k,v in other.items() if k!='validation_true_worst_group'}
    np.testing.assert_array_equal(predict_mlp(a['inferred_worst_group'][0],x[80:]),predict_mlp(b['inferred_worst_group'][0],x[80:]))
    for key, value in saved.items():
        torch.testing.assert_close(erm.state_dict()[key],value)


def test_guard_prefers_eligible_and_falls_back_to_closest_average():
    def rank(average,worst):
        return selection_rank('inferred_worst_group',{'average':average,'inferred_worst_group':worst},.88)
    assert rank(.88,.5)>rank(.879,.99)
    assert rank(.90,.7)>rank(.92,.6)
    assert rank(.87,.1)>rank(.86,.9)
    assert rank(.90,.7)>rank(.89,.7)


def test_validation_groups_cross_cluster_and_label_without_dro_merge():
    groups=cross_labels(np.array([0,0,1,1]),np.array([0,1,0,1]))
    assert len(np.unique(groups))==4
    correct=np.array([True,True,False,True])
    assert worst_accuracy(correct,groups)==0
    assert worst_accuracy(correct,np.array(['a','a','Unknown','b']))==1


def test_surrogate_selects_better_validation_f1_from_both_candidates():
    rng=np.random.default_rng(9)
    x=rng.normal(size=(500,3)); target=(x[:,0]+x[:,1]>2).astype(int)
    xv=rng.normal(size=(200,3)); truth=(xv[:,0]+xv[:,1]>2).astype(int)
    tree,meta=choose_surrogate(x,target,xv,truth,17)
    from sklearn.metrics import f1_score
    selected=f1_score(truth,tree.predict(xv),zero_division=0)
    assert selected==max(meta['fidelity_f1_balanced'],meta['fidelity_f1_unbalanced'])
    assert tree.get_depth()<=3 and meta['surrogate_selection_split']=='validation'


def test_oracle_sanity_reports_failure_and_success_without_hiding_it():
    from fairgroups.hmda_reporting import oracle_sanity
    rows=[]
    for seed in [17,29]:
        rows.extend([{'seed':seed,'method':'ERM-MLP','selection':'average','worst_group_accuracy':.8},
                     {'seed':seed,'method':'GDRO-oracle','selection':'inferred_worst_group','worst_group_accuracy':.7}])
    frame=pd.DataFrame(rows);_,passed,gain=oracle_sanity(frame)
    assert not passed['cy'] and gain['cy']<0
    frame.loc[frame.method.eq('GDRO-oracle'),'worst_group_accuracy']=.81
    _,passed,gain=oracle_sanity(frame)
    assert passed['cy'] and gain['cy']>0


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
import time
start = time.perf_counter()
subprocess.check_call([sys.executable, 'run_pilot.py', '--quick' if QUICK else '--full'])
print(f'Wall time: {time.perf_counter()-start:.1f} seconds')
subprocess.check_call([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown(Path('results/hmda_pilot_summary.md').read_text()))
for path in sorted(Path('results/figures').glob('*.png')):
    display(Image(filename=str(path)))


In [ ]:
# ดาวน์โหลดผลรวม (ไม่รวม data และ mlruns ที่มีขนาดใหญ่)
import shutil
shutil.make_archive('pilot_results', 'zip', 'results')
try:
    from google.colab import files
    files.download('pilot_results.zip')
except ImportError:
    print(Path('pilot_results.zip').resolve())


In [ ]:
# หน้าสาธิตท้าย notebook: ต้นแบบเพื่อการวิจัยเท่านั้น
import sys
sys.path.insert(0, str(BASE / 'src'))
from fairgroups.demo import build_interface
demo = build_interface(BASE / 'results')
demo.launch(share=True)
